In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_5roi_whitepaper_strict_validation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 64
RANDOM_STATE = 42
TEST_SIZE = 0.2
ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

# try all of these grouping strategies
GROUPING_MODES = [
    "session",
    "table_type",
    "physical_point",
    "table_type__physical_point",
]

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)

    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)

def choose_groups(df_sub, mode):
    if mode == "session":
        groups = df_sub["session"].astype(str)

    elif mode == "table_type":
        groups = df_sub["table_type"].astype(str)

    elif mode == "physical_point":
        # fallback if missing
        physical = df_sub["physical_point"].fillna("").astype(str)
        fallback = (
            df_sub["table_type"].fillna("").astype(str) + "__" +
            df_sub["session"].fillna("").astype(str)
        )
        groups = np.where(physical.str.len() > 0, physical, fallback)

    elif mode == "table_type__physical_point":
        physical = df_sub["physical_point"].fillna("").astype(str)
        fallback = df_sub["session"].fillna("").astype(str)
        groups = (
            df_sub["table_type"].fillna("").astype(str) + "__" +
            np.where(physical.str.len() > 0, physical, fallback)
        )

    else:
        raise ValueError(f"Unknown grouping mode: {mode}")

    return pd.Series(groups, index=df_sub.index)

# =========================================================
# BUILD IMAGE-LEVEL 5ROI DATASET
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

grouped = ann.groupby("image_path")
rows = []

for image_path, g in grouped:
    if set(g["roi_name"].tolist()) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        rec = {
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": safe_str(g.iloc[0]["session"]),
            "table_type": safe_str(g.iloc[0]["table_type"]),
            "point": safe_str(g.iloc[0]["point"]),
            "physical_point": safe_str(g.iloc[0]["physical_point"]),
            "surface": safe_str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
        }

        for _, row in g.iterrows():
            roi = row["roi_name"]
            x = int(row["x"])
            y = int(row["y"])
            rec[f"x_{roi}"] = x
            rec[f"y_{roi}"] = y
            rec[f"lux_{roi}"] = float(row["lux"])

            feats = compute_patch_features(img, x, y, ROI_SIZE)
            for k, v in feats.items():
                rec[f"{roi}_{k}"] = v

        rows.append(rec)

    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)

# white paper only
df = df[df["surface"] == "white_paper"].copy().reset_index(drop=True)

print("White paper rows:", len(df))
print("Unique sessions:", df["session"].nunique())
print("Unique table types:", df["table_type"].nunique())
print("Unique physical points (non-empty):", (df["physical_point"].fillna("").astype(str).str.len() > 0).sum())

feature_cols = []
for roi in ROI_ORDER:
    for feat in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]:
        feature_cols.append(f"{roi}_{feat}")

# =========================================================
# RUN STRICT VALIDATION
# =========================================================
all_metrics = []
all_predictions = {}

for grouping_mode in GROUPING_MODES:
    print(f"\n=== GROUPING MODE: {grouping_mode} ===")

    groups = choose_groups(df, grouping_mode)
    n_groups = pd.Series(groups).nunique()
    print("Rows:", len(df))
    print("Unique groups:", n_groups)

    if n_groups < 3:
        print("Skipping: too few unique groups.")
        continue

    X = df[feature_cols].values
    y = df["target_lux"].values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    try:
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))
    except Exception as e:
        print("Split failed:", e)
        continue

    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df["target_lux"].values
    X_test = test_df[feature_cols].values
    y_test = test_df["target_lux"].values

    model = ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mape_percent(y_test, y_pred)

    metrics_row = {
        "subset": "white_paper_only",
        "model": "ExtraTrees_5ROI",
        "grouping_mode": grouping_mode,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "n_groups_total": n_groups,
        "n_train_groups": pd.Series(groups.iloc[train_idx]).nunique(),
        "n_test_groups": pd.Series(groups.iloc[test_idx]).nunique(),
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }
    all_metrics.append(metrics_row)

    pred_df = test_df.copy()
    pred_df["pred"] = y_pred
    pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
    pred_df["residual"] = y_pred - pred_df["target_lux"].values
    pred_df["pct_err"] = np.abs((pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)) * 100.0
    pred_df["grouping_mode"] = grouping_mode

    pred_path = OUT_DIR / f"predictions__{grouping_mode}.csv"
    pred_df.to_csv(pred_path, index=False)
    all_predictions[grouping_mode] = pred_df

    print(pd.DataFrame([metrics_row]).to_string(index=False))

# =========================================================
# SAVE SUMMARY
# =========================================================
metrics_df = pd.DataFrame(all_metrics).sort_values("mae").reset_index(drop=True)
metrics_csv = OUT_DIR / "strict_validation_metrics.csv"
metrics_df.to_csv(metrics_csv, index=False)

print("\n========== STRICT VALIDATION SUMMARY ==========")
print(metrics_df.to_string(index=False))
print("\nSaved:", metrics_csv)

# =========================================================
# PLOT COMPARISON
# =========================================================
if len(metrics_df) > 0:
    plt.figure(figsize=(8, 5))
    plt.bar(metrics_df["grouping_mode"], metrics_df["mae"])
    plt.ylabel("MAE (lux)")
    plt.title("White Paper — ExtraTrees 5ROI — MAE by Grouping Strategy")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "mae_by_grouping_mode.png", dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.bar(metrics_df["grouping_mode"], metrics_df["mape_percent"])
    plt.ylabel("MAPE (%)")
    plt.title("White Paper — ExtraTrees 5ROI — MAPE by Grouping Strategy")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "mape_by_grouping_mode.png", dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.bar(metrics_df["grouping_mode"], metrics_df["r2"])
    plt.ylabel("R²")
    plt.title("White Paper — ExtraTrees 5ROI — R² by Grouping Strategy")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "r2_by_grouping_mode.png", dpi=200, bbox_inches="tight")
    plt.show()

print("\nAll outputs saved to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_5roi_whitepaper_strict_validation")
OUT_DIR = BASE_DIR / "plots_detailed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 64
ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
GROUPING_MODES = [
    "session",
    "physical_point",
    "table_type__physical_point",
    "table_type",
]

# =========================================================
# HELPERS
# =========================================================
def draw_example(row, title="", out_path=None):
    img = Image.open(row["image_path"])
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    for roi_name in ROI_ORDER:
        x = int(row[f"x_{roi_name}"])
        y = int(row[f"y_{roi_name}"])
        r = ROI_SIZE // 2
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")

    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"{title}\n"
        f"true={row['target_lux']:.1f}, pred={row['pred']:.1f}, abs_err={row['abs_err']:.1f}, pct_err={row['pct_err']:.2f}%\n"
        f"{Path(row['image_path']).name}",
        fontsize=10
    )
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

# =========================================================
# LOAD ALL PREDICTIONS
# =========================================================
preds = {}

for mode in GROUPING_MODES:
    p = BASE_DIR / f"predictions__{mode}.csv"
    if p.exists():
        df = pd.read_csv(p)
        preds[mode] = df
        print(f"Loaded {mode}: {len(df)} rows")
    else:
        print(f"Missing file for {mode}: {p}")

# =========================================================
# PER-GROUPING PLOTS
# =========================================================
for mode, df in preds.items():
    true = df["target_lux"].values
    pred = df["pred"].values
    abs_err = df["abs_err"].values
    resid = df["residual"].values
    pct_err = df["pct_err"].values

    # predicted vs true
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.6)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(f"Predicted vs True — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__pred_vs_true.png", dpi=200, bbox_inches="tight")
    plt.show()

    # absolute error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, abs_err, alpha=0.6)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(f"Absolute Error vs True — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__abs_err_vs_true.png", dpi=200, bbox_inches="tight")
    plt.show()

    # percentage error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, pct_err, alpha=0.6)
    plt.xlabel("True lux")
    plt.ylabel("Absolute percentage error (%)")
    plt.title(f"Percentage Error vs True — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__pct_err_vs_true.png", dpi=200, bbox_inches="tight")
    plt.show()

    # residuals vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, resid, alpha=0.6)
    plt.axhline(0, linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Residual (pred - true)")
    plt.title(f"Residual Plot — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__residuals.png", dpi=200, bbox_inches="tight")
    plt.show()

    # absolute error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(abs_err, bins=20)
    plt.xlabel("Absolute error (lux)")
    plt.ylabel("Count")
    plt.title(f"Absolute Error Histogram — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__abs_err_hist.png", dpi=200, bbox_inches="tight")
    plt.show()

    # percentage error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(pct_err, bins=20)
    plt.xlabel("Absolute percentage error (%)")
    plt.ylabel("Count")
    plt.title(f"Percentage Error Histogram — {mode}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{mode}__pct_err_hist.png", dpi=200, bbox_inches="tight")
    plt.show()

    # best / median / worst
    df_sorted = df.sort_values("abs_err").reset_index(drop=True)
    best_row = df_sorted.iloc[0]
    median_row = df_sorted.iloc[len(df_sorted)//2]
    worst_row = df_sorted.iloc[-1]

    draw_example(best_row, f"{mode} | BEST", OUT_DIR / f"{mode}__BEST.png")
    draw_example(median_row, f"{mode} | MEDIAN", OUT_DIR / f"{mode}__MEDIAN.png")
    draw_example(worst_row, f"{mode} | WORST", OUT_DIR / f"{mode}__WORST.png")

# =========================================================
# COMBINED PREDICTED VS TRUE
# =========================================================
available_modes = [m for m in GROUPING_MODES if m in preds]

plt.figure(figsize=(16, 12))

for i, mode in enumerate(available_modes, start=1):
    df = preds[mode]
    true = df["target_lux"].values
    pred = df["pred"].values

    plt.subplot(2, 2, i)
    plt.scatter(true, pred, alpha=0.6)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(mode)

plt.tight_layout()
plt.savefig(OUT_DIR / "combined__pred_vs_true.png", dpi=200, bbox_inches="tight")
plt.show()

# =========================================================
# COMBINED ABS ERROR VS TRUE
# =========================================================
plt.figure(figsize=(16, 12))

for i, mode in enumerate(available_modes, start=1):
    df = preds[mode]
    true = df["target_lux"].values
    abs_err = df["abs_err"].values

    plt.subplot(2, 2, i)
    plt.scatter(true, abs_err, alpha=0.6)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(mode)

plt.tight_layout()
plt.savefig(OUT_DIR / "combined__abs_err_vs_true.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved detailed plots to:", OUT_DIR)

ExtraTree, square area

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_square_only_subsets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 64
EXTRA_MARGIN = 0   # try 0, 8, 16
RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

def compute_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_size=64, extra_margin=0):
    r = roi_size // 2
    xs = [p[0] for p in corner_points_xy]
    ys = [p[1] for p in corner_points_xy]
    left = int(min(xs) - r - extra_margin)
    right = int(max(xs) + r + extra_margin)
    top = int(min(ys) - r - extra_margin)
    bottom = int(max(ys) + r + extra_margin)
    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []
for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue
    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        corner_points_xy = []
        for roi in CORNER_ROIS:
            rr = g[g["roi_name"] == roi].iloc[0]
            corner_points_xy.append((int(rr["x"]), int(rr["y"])))

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, roi_size=ROI_SIZE, extra_margin=EXTRA_MARGIN
        )
        patch = crop_bbox(img, left, top, right, bottom)
        feats = compute_features_from_array(np.asarray(patch))

        rows.append({
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
            **{f"square_{k}": v for k, v in feats.items()}
        })
    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))

feature_cols = [f"square_{k}" for k in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]]

subsets = {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
}

all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = df_sub.iloc[train_idx][feature_cols].values
    y_train = df_sub.iloc[train_idx]["target_lux"].values
    X_test = df_sub.iloc[test_idx][feature_cols].values
    y_test = df_sub.iloc[test_idx]["target_lux"].values

    model = ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    row = {
        "subset": subset_name,
        "model": "ExtraTrees_square_only",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(row)
    print(pd.DataFrame([row]).to_string(index=False))

metrics_df = pd.DataFrame(all_metrics).sort_values("mae")
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)
print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

Square area + cenral ROI

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_square_plus_center_subsets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 64
EXTRA_MARGIN = 0
RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

def compute_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_size=64, extra_margin=0):
    r = roi_size // 2
    xs = [p[0] for p in corner_points_xy]
    ys = [p[1] for p in corner_points_xy]
    left = int(min(xs) - r - extra_margin)
    right = int(max(xs) + r + extra_margin)
    top = int(min(ys) - r - extra_margin)
    bottom = int(max(ys) + r + extra_margin)
    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size
    return crop_bbox(img, left, top, right, bottom)

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []
for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue
    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        corner_points_xy = []
        for roi in CORNER_ROIS:
            rr = g[g["roi_name"] == roi].iloc[0]
            corner_points_xy.append((int(rr["x"]), int(rr["y"])))

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, roi_size=ROI_SIZE, extra_margin=EXTRA_MARGIN
        )
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_feats = compute_features_from_array(np.asarray(square_patch))

        c_row = g[g["roi_name"] == "C"].iloc[0]
        c_patch = crop_patch(img, int(c_row["x"]), int(c_row["y"]), ROI_SIZE)
        c_feats = compute_features_from_array(np.asarray(c_patch))

        rows.append({
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(c_row["lux"]),
            **{f"square_{k}": v for k, v in square_feats.items()},
            **{f"C_{k}": v for k, v in c_feats.items()},
        })
    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))

feature_cols = (
    [f"square_{k}" for k in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]] +
    [f"C_{k}" for k in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]]
)

subsets = {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
}

all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = df_sub.iloc[train_idx][feature_cols].values
    y_train = df_sub.iloc[train_idx]["target_lux"].values
    X_test = df_sub.iloc[test_idx][feature_cols].values
    y_test = df_sub.iloc[test_idx]["target_lux"].values

    model = ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    row = {
        "subset": subset_name,
        "model": "ExtraTrees_square_plus_center",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(row)
    print(pd.DataFrame([row]).to_string(index=False))

metrics_df = pd.DataFrame(all_metrics).sort_values("mae")
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)
print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

Square area + 5 ROI

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_square_plus_5roi_subsets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 64
EXTRA_MARGIN = 0
RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

def compute_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_size=64, extra_margin=0):
    r = roi_size // 2
    xs = [p[0] for p in corner_points_xy]
    ys = [p[1] for p in corner_points_xy]
    left = int(min(xs) - r - extra_margin)
    right = int(max(xs) + r + extra_margin)
    top = int(min(ys) - r - extra_margin)
    bottom = int(max(ys) + r + extra_margin)
    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size
    return crop_bbox(img, left, top, right, bottom)

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []
for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue
    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        corner_points_xy = []
        for roi in CORNER_ROIS:
            rr = g[g["roi_name"] == roi].iloc[0]
            corner_points_xy.append((int(rr["x"]), int(rr["y"])))

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, roi_size=ROI_SIZE, extra_margin=EXTRA_MARGIN
        )
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_feats = compute_features_from_array(np.asarray(square_patch))

        rec = {
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
            **{f"square_{k}": v for k, v in square_feats.items()},
        }

        for _, row in g.iterrows():
            roi = row["roi_name"]
            patch = crop_patch(img, int(row["x"]), int(row["y"]), ROI_SIZE)
            feats = compute_features_from_array(np.asarray(patch))
            for k, v in feats.items():
                rec[f"{roi}_{k}"] = v

        rows.append(rec)

    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))

feature_cols = [f"square_{k}" for k in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]]
for roi in ROI_ORDER:
    for feat in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]:
        feature_cols.append(f"{roi}_{feat}")

subsets = {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
}

all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = df_sub.iloc[train_idx][feature_cols].values
    y_train = df_sub.iloc[train_idx]["target_lux"].values
    X_test = df_sub.iloc[test_idx][feature_cols].values
    y_test = df_sub.iloc[test_idx]["target_lux"].values

    model = ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    row = {
        "subset": subset_name,
        "model": "ExtraTrees_square_plus_5roi",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(row)
    print(pd.DataFrame([row]).to_string(index=False))

metrics_df = pd.DataFrame(all_metrics).sort_values("mae")
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)
print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3")
OUT_DIR = ROOT / "compare_square_roi_models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD METRIC FILES
# =========================================================
files = {
    "square_only": ROOT / "extratrees_square_only_subsets" / "metrics.csv",
    "square_plus_center": ROOT / "extratrees_square_plus_center_subsets" / "metrics.csv",
    "square_plus_5roi": ROOT / "extratrees_square_plus_5roi_subsets" / "metrics.csv",
    "roi5_only": ROOT / "extratrees_5roi_subsets" / "extratrees_5roi_subsets_metrics.csv",
}

dfs = []
for label, path in files.items():
    if path.exists():
        df = pd.read_csv(path).copy()
        df["experiment"] = label
        dfs.append(df)
        print(f"Loaded: {label} -> {path}")
    else:
        print(f"Missing: {label} -> {path}")

metrics_df = pd.concat(dfs, ignore_index=True)

# standardize names just in case
metrics_df["subset"] = metrics_df["subset"].astype(str)
metrics_df["experiment"] = metrics_df["experiment"].astype(str)

# prettier labels
experiment_label_map = {
    "square_only": "Square only",
    "square_plus_center": "Square + Center ROI",
    "square_plus_5roi": "Square + 5 ROI",
    "roi5_only": "5 ROI only",
}
metrics_df["experiment_label"] = metrics_df["experiment"].map(experiment_label_map)

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]
subset_label_map = {
    "white_paper_only": "White paper only",
    "tables_only": "Tables only",
    "white_plus_tables": "White paper + tables",
    "all_5roi_data": "All 5-ROI data",
}
metrics_df["subset_label"] = metrics_df["subset"].map(subset_label_map)

# save combined table
metrics_df.to_csv(OUT_DIR / "combined_metrics.csv", index=False)

print("\n========== COMBINED METRICS ==========")
print(metrics_df[[
    "subset", "experiment", "mae", "rmse", "r2", "mape_percent"
]].sort_values(["subset", "mae"]).to_string(index=False))

# =========================================================
# PLOT FUNCTION
# =========================================================
def plot_metric_by_subset(metric_name, ylabel, better="lower"):
    for subset in subset_order:
        sub = metrics_df[metrics_df["subset"] == subset].copy()
        if len(sub) == 0:
            continue

        order = ["square_only", "square_plus_center", "square_plus_5roi", "roi5_only"]
        sub["experiment"] = pd.Categorical(sub["experiment"], categories=order, ordered=True)
        sub = sub.sort_values("experiment")

        plt.figure(figsize=(8, 5))
        plt.bar(sub["experiment_label"], sub[metric_name])
        plt.ylabel(ylabel)
        plt.title(f"{ylabel} — {subset_label_map.get(subset, subset)}")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{subset}__{metric_name}.png", dpi=200, bbox_inches="tight")
        plt.show()

def plot_all_subsets_for_metric(metric_name, ylabel):
    subsets_present = [s for s in subset_order if s in metrics_df["subset"].unique()]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, subset in zip(axes, subsets_present):
        sub = metrics_df[metrics_df["subset"] == subset].copy()
        order = ["square_only", "square_plus_center", "square_plus_5roi", "roi5_only"]
        sub["experiment"] = pd.Categorical(sub["experiment"], categories=order, ordered=True)
        sub = sub.sort_values("experiment")

        ax.bar(sub["experiment_label"], sub[metric_name])
        ax.set_title(subset_label_map.get(subset, subset))
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=20)

    # hide unused panels if any
    for i in range(len(subsets_present), 4):
        axes[i].axis("off")

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"combined__{metric_name}.png", dpi=200, bbox_inches="tight")
    plt.show()

# =========================================================
# MAKE PLOTS
# =========================================================
plot_metric_by_subset("mae", "MAE (lux)")
plot_metric_by_subset("rmse", "RMSE (lux)")
plot_metric_by_subset("r2", "R²")
plot_metric_by_subset("mape_percent", "MAPE (%)")

plot_all_subsets_for_metric("mae", "MAE (lux)")
plot_all_subsets_for_metric("rmse", "RMSE (lux)")
plot_all_subsets_for_metric("r2", "R²")
plot_all_subsets_for_metric("mape_percent", "MAPE (%)")

print("\nSaved plots to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/compare_square_roi_models")
df = pd.read_csv(ROOT / "combined_metrics.csv")

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]
subset_label_map = {
    "white_paper_only": "White paper",
    "tables_only": "Tables",
    "white_plus_tables": "White + tables",
    "all_5roi_data": "All 5-ROI data",
}
experiment_order = ["square_only", "square_plus_center", "square_plus_5roi", "roi5_only"]
experiment_label_map = {
    "square_only": "Square only",
    "square_plus_center": "Square + center",
    "square_plus_5roi": "Square + 5 ROI",
    "roi5_only": "5 ROI only",
}

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(subset_order))
width = 0.2

for i, exp in enumerate(experiment_order):
    vals = []
    for subset in subset_order:
        sub = df[(df["subset"] == subset) & (df["experiment"] == exp)]
        vals.append(sub["mae"].iloc[0] if len(sub) else np.nan)

    ax.bar(x + (i - 1.5) * width, vals, width, label=experiment_label_map[exp])

ax.set_xticks(x)
ax.set_xticklabels([subset_label_map[s] for s in subset_order])
ax.set_ylabel("MAE (lux)")
ax.set_title("Comparison of ExtraTrees feature configurations")
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "poster_summary__mae_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3")
OUT_DIR = ROOT / "compare_square_roi_models_visuals"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# load metric files
# -----------------------------
files = {
    "square_only": ROOT / "extratrees_square_only_subsets" / "metrics.csv",
    "square_plus_center": ROOT / "extratrees_square_plus_center_subsets" / "metrics.csv",
    "square_plus_5roi": ROOT / "extratrees_square_plus_5roi_subsets" / "metrics.csv",
    "roi5_only": ROOT / "extratrees_5roi_subsets" / "extratrees_5roi_subsets_metrics.csv",
}

dfs = []
for exp, path in files.items():
    if path.exists():
        df = pd.read_csv(path).copy()
        df["experiment"] = exp
        dfs.append(df)

metrics_df = pd.concat(dfs, ignore_index=True)

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]
subset_label_map = {
    "white_paper_only": "White paper",
    "tables_only": "Tables",
    "white_plus_tables": "White + tables",
    "all_5roi_data": "All 5-ROI data",
}
experiment_order = ["square_only", "square_plus_center", "square_plus_5roi", "roi5_only"]
experiment_label_map = {
    "square_only": "Square only",
    "square_plus_center": "Square + center",
    "square_plus_5roi": "Square + 5 ROI",
    "roi5_only": "5 ROI only",
}

# -----------------------------
# grouped metric plots
# -----------------------------
def grouped_metric_plot(metric_name, ylabel, filename):
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(subset_order))
    width = 0.2

    for i, exp in enumerate(experiment_order):
        vals = []
        for subset in subset_order:
            sub = metrics_df[(metrics_df["subset"] == subset) & (metrics_df["experiment"] == exp)]
            vals.append(sub[metric_name].iloc[0] if len(sub) else np.nan)

        ax.bar(x + (i - 1.5) * width, vals, width, label=experiment_label_map[exp])

    ax.set_xticks(x)
    ax.set_xticklabels([subset_label_map[s] for s in subset_order])
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} comparison across feature configurations")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

grouped_metric_plot("mae", "MAE (lux)", "mae_comparison.png")
grouped_metric_plot("rmse", "RMSE (lux)", "rmse_comparison.png")
grouped_metric_plot("r2", "R²", "r2_comparison.png")
grouped_metric_plot("mape_percent", "MAPE (%)", "mape_comparison.png")

# -----------------------------
# load prediction files for best model comparison
# -----------------------------
pred_files = {
    "roi5_only": ROOT / "extratrees_5roi_subsets",
    "square_plus_5roi": ROOT / "extratrees_square_plus_5roi_subsets",
}

# note:
# roi5_only predictions were saved per subset as:
#   all_5roi_data__predictions.csv, etc.
# square_plus_5roi script as provided only saved metrics.
# if you do not yet have predictions for square_plus_5roi, re-run that script with prediction saving,
# or use the extra block below to regenerate them.

# -----------------------------
# comparison plots if prediction files exist
# -----------------------------
for subset in subset_order:
    roi5_pred_path = ROOT / "extratrees_5roi_subsets" / f"{subset}__predictions.csv"
    sq5_pred_path = ROOT / "extratrees_square_plus_5roi_subsets" / f"{subset}__predictions.csv"

    if roi5_pred_path.exists() and sq5_pred_path.exists():
        df1 = pd.read_csv(roi5_pred_path)
        df2 = pd.read_csv(sq5_pred_path)

        # predicted vs true
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        for ax, dfp, title in zip(
            axes,
            [df1, df2],
            ["5 ROI only", "Square + 5 ROI"]
        ):
            true = dfp["target_lux"].values
            pred = dfp["pred"].values
            mn = min(true.min(), pred.min())
            mx = max(true.max(), pred.max())

            ax.scatter(true, pred, alpha=0.6)
            ax.plot([mn, mx], [mn, mx], linestyle="--")
            ax.set_xlabel("True lux")
            ax.set_ylabel("Predicted lux")
            ax.set_title(title)

        fig.suptitle(f"Predicted vs True — {subset_label_map[subset]}")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{subset}__pred_vs_true__roi5_vs_sq5.png", dpi=220, bbox_inches="tight")
        plt.show()

        # abs error vs true
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        for ax, dfp, title in zip(
            axes,
            [df1, df2],
            ["5 ROI only", "Square + 5 ROI"]
        ):
            ax.scatter(dfp["target_lux"], dfp["abs_err"], alpha=0.6)
            ax.set_xlabel("True lux")
            ax.set_ylabel("Absolute error (lux)")
            ax.set_title(title)

        fig.suptitle(f"Absolute Error vs True — {subset_label_map[subset]}")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{subset}__abs_err_vs_true__roi5_vs_sq5.png", dpi=220, bbox_inches="tight")
        plt.show()

print("Saved visual comparison plots to:", OUT_DIR)

ROI size experiment

ExtraTrees for 1 ROI at 64, 128, 256

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_roi1_size_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZES = [64, 128, 256]
RANDOM_STATE = 42
TEST_SIZE = 0.2
ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_features_from_patch(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

# build image-level dataset from 5ROI images, but using only central ROI features
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

grouped = ann.groupby("image_path")
base_rows = []

for image_path, g in grouped:
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    c_row = g[g["roi_name"] == "C"].iloc[0]

    base_rows.append({
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(c_row["lux"]),
        "x_C": int(c_row["x"]),
        "y_C": int(c_row["y"]),
    })

base_df = pd.DataFrame(base_rows)
print("Base image-level rows:", len(base_df))

subsets_func = lambda df: {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
}

all_metrics = []

for roi_size in ROI_SIZES:
    print(f"\n================ ROI SIZE = {roi_size} ================\n")

    rows = []
    for _, row in base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")
            feats = compute_features_from_patch(img, row["x_C"], row["y_C"], roi_size)

            rows.append({
                **row.to_dict(),
                "roi_size": roi_size,
                **{f"C_{k}": v for k, v in feats.items()}
            })
        except Exception as e:
            print("Skipped:", row["image_path"], e)

    df = pd.DataFrame(rows)
    feature_cols = [f"C_{k}" for k in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]]

    subsets = subsets_func(df)

    for subset_name, df_sub in subsets.items():
        print(f"=== {subset_name} ===")
        print("Rows:", len(df_sub))
        if len(df_sub) < 20:
            continue

        X = df_sub[feature_cols].values
        y = df_sub["target_lux"].values
        groups = df_sub["session"].astype(str).values

        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))

        X_train = df_sub.iloc[train_idx][feature_cols].values
        y_train = df_sub.iloc[train_idx]["target_lux"].values
        X_test = df_sub.iloc[test_idx][feature_cols].values
        y_test = df_sub.iloc[test_idx]["target_lux"].values

        model = ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "subset": subset_name,
            "model": "ExtraTrees_roi1_only",
            "roi_size": roi_size,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_metrics.append(result)
        print(pd.DataFrame([result]).to_string(index=False))

metrics_df = pd.DataFrame(all_metrics).sort_values(["subset", "mae"]).reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics_roi1_sizes.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_roi1_sizes.csv")

ExtraTrees for 5 ROI at 64, 128, 256

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_roi5_size_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_SIZES = [64, 128, 256]
RANDOM_STATE = 42
TEST_SIZE = 0.2
ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_features_from_patch(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

grouped = ann.groupby("image_path")
base_rows = []

for image_path, g in grouped:
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    rec = {
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
    }

    for _, row in g.iterrows():
        roi = row["roi_name"]
        rec[f"x_{roi}"] = int(row["x"])
        rec[f"y_{roi}"] = int(row["y"])

    base_rows.append(rec)

base_df = pd.DataFrame(base_rows)
print("Base image-level rows:", len(base_df))

subsets_func = lambda df: {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
}

all_metrics = []

for roi_size in ROI_SIZES:
    print(f"\n================ ROI SIZE = {roi_size} ================\n")

    rows = []
    for _, row in base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = {**row.to_dict(), "roi_size": roi_size}

            for roi in ROI_ORDER:
                feats = compute_features_from_patch(img, row[f"x_{roi}"], row[f"y_{roi}"], roi_size)
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v

            rows.append(rec)
        except Exception as e:
            print("Skipped:", row["image_path"], e)

    df = pd.DataFrame(rows)

    feature_cols = []
    for roi in ROI_ORDER:
        for feat in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]:
            feature_cols.append(f"{roi}_{feat}")

    subsets = subsets_func(df)

    for subset_name, df_sub in subsets.items():
        print(f"=== {subset_name} ===")
        print("Rows:", len(df_sub))
        if len(df_sub) < 20:
            continue

        X = df_sub[feature_cols].values
        y = df_sub["target_lux"].values
        groups = df_sub["session"].astype(str).values

        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))

        X_train = df_sub.iloc[train_idx][feature_cols].values
        y_train = df_sub.iloc[train_idx]["target_lux"].values
        X_test = df_sub.iloc[test_idx][feature_cols].values
        y_test = df_sub.iloc[test_idx]["target_lux"].values

        model = ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "subset": subset_name,
            "model": "ExtraTrees_roi5_only",
            "roi_size": roi_size,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_metrics.append(result)
        print(pd.DataFrame([result]).to_string(index=False))

metrics_df = pd.DataFrame(all_metrics).sort_values(["subset", "mae"]).reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics_roi5_sizes.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_roi5_sizes.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3")
OUT_DIR = ROOT / "roi_size_comparison_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df1 = pd.read_csv(ROOT / "extratrees_roi1_size_comparison" / "metrics_roi1_sizes.csv")
df5 = pd.read_csv(ROOT / "extratrees_roi5_size_comparison" / "metrics_roi5_sizes.csv")

df1["config"] = "1 ROI"
df5["config"] = "5 ROI"

df = pd.concat([df1, df5], ignore_index=True)

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]
subset_label_map = {
    "white_paper_only": "White paper",
    "tables_only": "Tables",
    "white_plus_tables": "White + tables",
    "all_5roi_data": "All 5-ROI data",
}

# 1) one plot per subset: MAE by ROI size
for subset in subset_order:
    sub = df[df["subset"] == subset].copy()
    if len(sub) == 0:
        continue

    plt.figure(figsize=(8, 5))

    for config in ["1 ROI", "5 ROI"]:
        ss = sub[sub["config"] == config].sort_values("roi_size")
        plt.plot(ss["roi_size"], ss["mae"], marker="o", label=config)

    plt.xlabel("ROI size (px)")
    plt.ylabel("MAE (lux)")
    plt.title(f"MAE vs ROI size — {subset_label_map[subset]}")
    plt.xticks([64, 128, 256])
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{subset}__mae_vs_roi_size.png", dpi=220, bbox_inches="tight")
    plt.show()

# 2) one combined figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, subset in zip(axes, subset_order):
    sub = df[df["subset"] == subset].copy()

    for config in ["1 ROI", "5 ROI"]:
        ss = sub[sub["config"] == config].sort_values("roi_size")
        ax.plot(ss["roi_size"], ss["mae"], marker="o", label=config)

    ax.set_title(subset_label_map[subset])
    ax.set_xlabel("ROI size (px)")
    ax.set_ylabel("MAE (lux)")
    ax.set_xticks([64, 128, 256])

axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "combined__mae_vs_roi_size.png", dpi=220, bbox_inches="tight")
plt.show()

# 3) summary table of best size per subset
best_rows = df.sort_values("mae").groupby(["subset", "config"], as_index=False).first()
best_rows.to_csv(OUT_DIR / "best_roi_size_summary.csv", index=False)

print("Saved plots to:", OUT_DIR)
print("\nBest per subset/config:")
print(best_rows[["subset", "config", "roi_size", "mae", "rmse", "r2", "mape_percent"]].to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_roi5_mixed_size_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

# ---------------------------------------------------------
# Configurations to test
# ---------------------------------------------------------
CONFIGS = [
    {"config_name": "all128", "C": 128, "UL": 128, "UR": 128, "LR": 128, "LL": 128},
    {"config_name": "C256_others64", "C": 256, "UL": 64, "UR": 64, "LR": 64, "LL": 64},
    {"config_name": "C256_others128", "C": 256, "UL": 128, "UR": 128, "LR": 128, "LL": 128},
]

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_features_from_patch(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)

    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

# ---------------------------------------------------------
# Build base image-level dataset
# ---------------------------------------------------------
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

grouped = ann.groupby("image_path")
base_rows = []

for image_path, g in grouped:
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    rec = {
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
    }

    for _, row in g.iterrows():
        roi = row["roi_name"]
        rec[f"x_{roi}"] = int(row["x"])
        rec[f"y_{roi}"] = int(row["y"])

    base_rows.append(rec)

base_df = pd.DataFrame(base_rows)
print("Base image-level rows:", len(base_df))

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

all_metrics = []

# ---------------------------------------------------------
# Run each mixed-size configuration
# ---------------------------------------------------------
for cfg in CONFIGS:
    print(f"\n================ CONFIG = {cfg['config_name']} ================\n")

    rows = []
    for _, row in base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = {**row.to_dict(), "config_name": cfg["config_name"]}

            for roi in ROI_ORDER:
                roi_size = cfg[roi]
                feats = compute_features_from_patch(img, row[f"x_{roi}"], row[f"y_{roi}"], roi_size)
                rec[f"{roi}_roi_size"] = roi_size
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v

            rows.append(rec)

        except Exception as e:
            print("Skipped:", row["image_path"], e)

    df = pd.DataFrame(rows)

    feature_cols = []
    for roi in ROI_ORDER:
        for feat in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]:
            feature_cols.append(f"{roi}_{feat}")

    subsets = make_subsets(df)

    for subset_name, df_sub in subsets.items():
        print(f"=== {subset_name} ===")
        print("Rows:", len(df_sub))
        if len(df_sub) < 20:
            continue

        X = df_sub[feature_cols].values
        y = df_sub["target_lux"].values
        groups = df_sub["session"].astype(str).values

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE
        )
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))

        X_train = df_sub.iloc[train_idx][feature_cols].values
        y_train = df_sub.iloc[train_idx]["target_lux"].values
        X_test = df_sub.iloc[test_idx][feature_cols].values
        y_test = df_sub.iloc[test_idx]["target_lux"].values

        model = ExtraTreesRegressor(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "subset": subset_name,
            "model": "ExtraTrees_roi5_mixed_sizes",
            "config_name": cfg["config_name"],
            "C_size": cfg["C"],
            "other_size": cfg["UL"],  # UL/UR/LR/LL are same in these configs
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_metrics.append(result)
        print(pd.DataFrame([result]).to_string(index=False))

        pred_df = df_sub.iloc[test_idx].copy()
        pred_df["pred"] = y_pred
        pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
        pred_df["residual"] = y_pred - pred_df["target_lux"].values
        pred_df["pct_err"] = np.abs(
            (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
        ) * 100.0
        pred_df.to_csv(OUT_DIR / f"{cfg['config_name']}__{subset_name}__predictions.csv", index=False)

# ---------------------------------------------------------
# Save summary
# ---------------------------------------------------------
metrics_df = pd.DataFrame(all_metrics).sort_values(["subset", "mae"]).reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics_mixed_roi_sizes.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_mixed_roi_sizes.csv")

Different Train-Test split (Train dataset: all photos with 5 ROI, Test dataset: all photos with 1 ROI)

QA: show random 1-ROI photos with all 5 ROI drawn


In [ ]:
from pathlib import Path
import pandas as pd
import random
import math
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

# fixed coordinates after EXIF transpose
ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZE = 64
N_IMAGES = 6
SEED = 42

random.seed(SEED)

ann = pd.read_csv(ANNOT_CSV)

# 1-ROI photos = center_only_lc or center_only_lt
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

print("1-ROI rows:", len(ann1))

sample_df = ann1.sample(min(N_IMAGES, len(ann1)), random_state=SEED).reset_index(drop=True)

def draw_all_roi_on_image(image_path, roi_coords, roi_size=64):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    r = roi_size // 2
    for roi_name, (x, y) in roi_coords.items():
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")

    return img

cols = 2
rows = math.ceil(len(sample_df) / cols)

plt.figure(figsize=(12, 6 * rows))

for i, row in sample_df.iterrows():
    img_overlay = draw_all_roi_on_image(row["image_path"], ROI_COORDS, ROI_SIZE)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_overlay)
    plt.axis("off")
    plt.title(
        f"{Path(row['image_path']).name}\n"
        f"surface={row['surface']} | lux(C)={row['lux']}",
        fontsize=9
    )

plt.tight_layout()
plt.show()

Train on 5-ROI photos, test on 1-ROI photos using real 5-ROI feature extraction



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/train5roi_test1roi_real5features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]

# fixed ROI coordinates after EXIF transpose
ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

CONFIGS = [
    {"config_name": "all64", "C": 64, "UL": 64, "UR": 64, "LR": 64, "LL": 64},
    {"config_name": "all128", "C": 128, "UL": 128, "UR": 128, "LR": 128, "LL": 128},
    {"config_name": "all256", "C": 256, "UL": 256, "UR": 256, "LR": 256, "LL": 256},
    {"config_name": "C256_others64", "C": 256, "UL": 64, "UR": 64, "LR": 64, "LL": 64},
    {"config_name": "C256_others128", "C": 256, "UL": 128, "UR": 128, "LR": 128, "LL": 128},
    {"config_name": "C256_others256", "C": 256, "UL": 256, "UR": 256, "LR": 256, "LL": 256},
]

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_features_from_patch(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)

    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
    }

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

ann = pd.read_csv(ANNOT_CSV)

# ---------------------------------------------------------
# TRAIN SET: true 5-ROI photos
# ---------------------------------------------------------
ann5 = ann[ann["label_type"] == "5roi"].copy()

train_rows_base = []
for image_path, g in ann5.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    rec = {
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
    }

    for _, row in g.iterrows():
        roi = row["roi_name"]
        rec[f"x_{roi}"] = int(row["x"])
        rec[f"y_{roi}"] = int(row["y"])

    train_rows_base.append(rec)

train_base_df = pd.DataFrame(train_rows_base)
print("Train rows (5ROI photos):", len(train_base_df))

# ---------------------------------------------------------
# TEST SET: 1-ROI photos, but full image available
# ---------------------------------------------------------
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

test_rows_base = []
for _, row in ann1.iterrows():
    rec = {
        "image_path": row["image_path"],
        "filename": row["filename"],
        "session": str(row["session"]),
        "table_type": str(row["table_type"]),
        "surface": str(row["surface"]).lower(),
        "target_lux": float(row["lux"]),  # only center lux label
    }

    for roi in ROI_ORDER:
        rec[f"x_{roi}"] = ROI_COORDS[roi][0]
        rec[f"y_{roi}"] = ROI_COORDS[roi][1]

    test_rows_base.append(rec)

test_base_df = pd.DataFrame(test_rows_base)
print("Test rows (1ROI photos, real 5-ROI feature extraction):", len(test_base_df))

all_results = []

for cfg in CONFIGS:
    print(f"\n================ CONFIG = {cfg['config_name']} ================\n")

    # ---------------- TRAIN FEATURES ----------------
    train_rows = []
    for _, row in train_base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = {**row.to_dict(), "config_name": cfg["config_name"]}

            for roi in ROI_ORDER:
                feats = compute_features_from_patch(
                    img,
                    row[f"x_{roi}"],
                    row[f"y_{roi}"],
                    cfg[roi]
                )
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v

            train_rows.append(rec)
        except Exception as e:
            print("Skipped train:", row["image_path"], e)

    train_df_full = pd.DataFrame(train_rows)

    # ---------------- TEST FEATURES ----------------
    test_rows = []
    for _, row in test_base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = {**row.to_dict(), "config_name": cfg["config_name"]}

            for roi in ROI_ORDER:
                feats = compute_features_from_patch(
                    img,
                    row[f"x_{roi}"],
                    row[f"y_{roi}"],
                    cfg[roi]
                )
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v

            test_rows.append(rec)
        except Exception as e:
            print("Skipped test:", row["image_path"], e)

    test_df_full = pd.DataFrame(test_rows)

    feature_cols = []
    for roi in ROI_ORDER:
        for feat in ["mean_r", "mean_g", "mean_b", "mean_luma", "std_luma"]:
            feature_cols.append(f"{roi}_{feat}")

    train_subsets = make_subsets(train_df_full)
    test_subsets = make_subsets(test_df_full)

    for subset_name in train_subsets.keys():
        train_df = train_subsets[subset_name].copy()
        test_df = test_subsets[subset_name].copy()

        print(f"=== {subset_name} ===")
        print("Train rows:", len(train_df), " Test rows:", len(test_df))

        if len(train_df) < 20 or len(test_df) < 20:
            print("Skipping: too few rows.")
            continue

        X_train = train_df[feature_cols].values
        y_train = train_df["target_lux"].values
        X_test = test_df[feature_cols].values
        y_test = test_df["target_lux"].values

        model = ExtraTreesRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "subset": subset_name,
            "model": "ExtraTrees_train5ROI_test1ROI_real5features",
            "config_name": cfg["config_name"],
            "C_size": cfg["C"],
            "other_size": cfg["UL"],
            "n_train": len(train_df),
            "n_test": len(test_df),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_results.append(result)
        print(pd.DataFrame([result]).to_string(index=False))

        pred_df = test_df.copy()
        pred_df["pred"] = y_pred
        pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
        pred_df["pct_err"] = np.abs(
            (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
        ) * 100.0
        pred_df.to_csv(OUT_DIR / f"{cfg['config_name']}__{subset_name}__predictions.csv", index=False)

results_df = pd.DataFrame(all_results).sort_values(["subset", "mae"]).reset_index(drop=True)
results_df.to_csv(OUT_DIR / "metrics_train5roi_test1roi_real5features.csv", index=False)

print("\n========== FINAL RESULTS ==========")
print(results_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_train5roi_test1roi_real5features.csv")

visualize random 1-ROI photos with all 5 ROI

In [ ]:
from pathlib import Path
import pandas as pd
import random
import math
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZE = 64
N_IMAGES = 12
SEED = 42

random.seed(SEED)

ann = pd.read_csv(ANNOT_CSV)
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

print("1-ROI rows:", len(ann1))

sample_df = ann1.sample(min(N_IMAGES, len(ann1)), random_state=SEED).reset_index(drop=True)

def draw_roi_overlay(image_path, roi_coords, roi_size=64):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    r = roi_size // 2
    for roi_name, (x, y) in roi_coords.items():
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")

    return img

cols = 3
rows = math.ceil(len(sample_df) / cols)

plt.figure(figsize=(16, 5 * rows))

for i, row in sample_df.iterrows():
    img_overlay = draw_roi_overlay(row["image_path"], ROI_COORDS, ROI_SIZE)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_overlay)
    plt.axis("off")
    plt.title(
        f"{Path(row['image_path']).name}\n"
        f"session={row['session']} | surface={row['surface']} | lux(C)={row['lux']}",
        fontsize=9
    )

plt.tight_layout()
plt.show()

visualize the oldest 1-ROI photos first

In [ ]:
from pathlib import Path
import pandas as pd
import math
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZE = 64
N_IMAGES = 24

ann = pd.read_csv(ANNOT_CSV)
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

# Try to sort by session number if possible
def session_num(s):
    s = str(s)
    digits = "".join(ch for ch in s if ch.isdigit())
    return int(digits) if digits else 999999

ann1["session_num"] = ann1["session"].apply(session_num)
oldest_df = ann1.sort_values(["session_num", "filename"]).head(N_IMAGES).reset_index(drop=True)

print(oldest_df[["session", "filename", "surface", "lux"]].to_string(index=False))

def draw_roi_overlay(image_path, roi_coords, roi_size=64):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    r = roi_size // 2
    for roi_name, (x, y) in roi_coords.items():
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")

    return img

cols = 3
rows = math.ceil(len(oldest_df) / cols)

plt.figure(figsize=(16, 5 * rows))

for i, row in oldest_df.iterrows():
    img_overlay = draw_roi_overlay(row["image_path"], ROI_COORDS, ROI_SIZE)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_overlay)
    plt.axis("off")
    plt.title(
        f"{Path(row['image_path']).name}\n"
        f"{row['session']} | {row['surface']} | C={row['lux']}",
        fontsize=9
    )

plt.tight_layout()
plt.show()

automatically flag suspicious 1-ROI photos

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/roi_qc_checks")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}
ROI_SIZE = 64

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def patch_mean_luma(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch).astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return float(luma.mean())

ann = pd.read_csv(ANNOT_CSV)
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

rows = []

for _, row in ann1.iterrows():
    try:
        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        rec = {
            "image_path": row["image_path"],
            "filename": row["filename"],
            "session": row["session"],
            "surface": row["surface"],
            "lux_C": row["lux"],
        }

        for roi_name, (x, y) in ROI_COORDS.items():
            rec[f"{roi_name}_mean_luma"] = patch_mean_luma(img, x, y, ROI_SIZE)

        corner_vals = [rec["UL_mean_luma"], rec["UR_mean_luma"], rec["LR_mean_luma"], rec["LL_mean_luma"]]
        rec["corner_mean_luma"] = float(np.mean(corner_vals))
        rec["corner_std_luma"] = float(np.std(corner_vals))
        rec["C_minus_corner_mean_luma"] = rec["C_mean_luma"] - rec["corner_mean_luma"]
        rec["max_corner_diff_luma"] = float(np.max(corner_vals) - np.min(corner_vals))

        rows.append(rec)

    except Exception as e:
        print("Skipped:", row["image_path"], e)

qc_df = pd.DataFrame(rows)

# crude suspiciousness score
qc_df["suspicious_score"] = (
    np.abs(qc_df["C_minus_corner_mean_luma"]) +
    qc_df["corner_std_luma"] * 2.0 +
    qc_df["max_corner_diff_luma"]
)

qc_df = qc_df.sort_values("suspicious_score", ascending=False).reset_index(drop=True)
qc_df.to_csv(OUT_DIR / "one_roi_qc_suspicious_ranked.csv", index=False)

print("Saved:", OUT_DIR / "one_roi_qc_suspicious_ranked.csv")
print("\nTop suspicious examples:")
print(qc_df.head(20)[[
    "session", "filename", "surface", "lux_C",
    "C_mean_luma", "corner_mean_luma", "corner_std_luma",
    "max_corner_diff_luma", "suspicious_score"
]].to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import math
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZE = 64
BAD_SESSIONS = {"267", "268", "269", "270", "F267", "F268", "F269", "F270"}

ann = pd.read_csv(ANNOT_CSV)
ann1 = ann[ann["label_type"].isin(["center_only_lc", "center_only_lt"])].copy()
ann1 = ann1[ann1["roi_name"] == "C"].copy()

ann1["session_str"] = ann1["session"].astype(str)
bad_df = ann1[ann1["session_str"].isin(BAD_SESSIONS)].copy().reset_index(drop=True)

print("Rows in bad sessions:", len(bad_df))
print(bad_df[["session", "filename", "surface", "lux"]].to_string(index=False))

def draw_overlay(image_path, roi_coords, roi_size=64):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    r = roi_size // 2
    for roi_name, (x, y) in roi_coords.items():
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")

    return img

cols = 3
rows = math.ceil(len(bad_df) / cols) if len(bad_df) > 0 else 1

plt.figure(figsize=(16, 5 * rows))

for i, row in bad_df.iterrows():
    img_overlay = draw_overlay(row["image_path"], ROI_COORDS, ROI_SIZE)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img_overlay)
    plt.axis("off")
    plt.title(
        f"{row['session']} | {row['surface']} | C={row['lux']}\n{Path(row['image_path']).name}",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/train5roi_test1roi_real5features")
METRICS_CSV = BASE_DIR / "metrics_train5roi_test1roi_real5features.csv"
OUT_DIR = BASE_DIR / "visualizations_existing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(METRICS_CSV)

print(df.to_string(index=False))

subset_order = ["all_data", "white_paper_only", "tables_only", "white_plus_tables"]
subset_label_map = {
    "all_data": "All data",
    "white_paper_only": "White paper",
    "tables_only": "Tables",
    "white_plus_tables": "White + tables",
}

config_order = [
    "all64",
    "all128",
    "all256",
    "C256_others64",
    "C256_others128",
    "C256_others256",
]
config_label_map = {
    "all64": "all64",
    "all128": "all128",
    "all256": "all256",
    "C256_others64": "C256/o64",
    "C256_others128": "C256/o128",
    "C256_others256": "C256/o256",
}

def grouped_metric_plot(metric_name, ylabel, filename):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, subset in zip(axes, subset_order):
        sub = df[df["subset"] == subset].copy()
        sub["config_name"] = pd.Categorical(sub["config_name"], categories=config_order, ordered=True)
        sub = sub.sort_values("config_name")

        ax.bar([config_label_map[c] for c in sub["config_name"]], sub[metric_name])
        ax.set_title(subset_label_map[subset])
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

grouped_metric_plot("mae", "MAE (lux)", "mae_by_config.png")
grouped_metric_plot("rmse", "RMSE (lux)", "rmse_by_config.png")
grouped_metric_plot("r2", "R²", "r2_by_config.png")
grouped_metric_plot("mape_percent", "MAPE (%)", "mape_by_config.png")

# one compact poster-style MAE figure
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(subset_order))
width = 0.13

for i, cfg in enumerate(config_order):
    vals = []
    for subset in subset_order:
        sub = df[(df["subset"] == subset) & (df["config_name"] == cfg)]
        vals.append(sub["mae"].iloc[0] if len(sub) else np.nan)
    ax.bar(x + (i - 2.5) * width, vals, width, label=config_label_map[cfg])

ax.set_xticks(x)
ax.set_xticklabels([subset_label_map[s] for s in subset_order])
ax.set_ylabel("MAE (lux)")
ax.set_title("Train on 5-ROI photos, test on 1-ROI photos")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "poster_mae_summary.png", dpi=220, bbox_inches="tight")
plt.show()

print("Saved plots to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/train5roi_test1roi_real5features")
OUT_DIR = BASE_DIR / "visualizations_existing_predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_files = sorted(BASE_DIR.glob("*__predictions.csv"))

print("Found prediction files:", len(pred_files))
for p in pred_files[:10]:
    print("-", p.name)

for p in pred_files:
    df = pd.read_csv(p)

    name = p.stem.replace("__predictions", "")
    if not {"target_lux", "pred", "abs_err", "pct_err"}.issubset(df.columns):
        print("Skipping (missing columns):", p.name)
        continue

    true = df["target_lux"].values
    pred = df["pred"].values
    abs_err = df["abs_err"].values
    pct_err = df["pct_err"].values

    # predicted vs true
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.5)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(f"Predicted vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pred_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # absolute error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, abs_err, alpha=0.5)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(f"Absolute Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # percentage error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, pct_err, alpha=0.5)
    plt.xlabel("True lux")
    plt.ylabel("Absolute percentage error (%)")
    plt.title(f"Percentage Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # histograms
    plt.figure(figsize=(6, 5))
    plt.hist(abs_err, bins=30)
    plt.xlabel("Absolute error (lux)")
    plt.ylabel("Count")
    plt.title(f"Absolute Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.hist(pct_err, bins=30)
    plt.xlabel("Absolute percentage error (%)")
    plt.ylabel("Count")
    plt.title(f"Percentage Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved prediction plots to:", OUT_DIR)

This tests a richer feature set built from:

per-ROI photometric features
per-ROI gradient / contrast / highlight-shadow proxies
center-to-corner relational features
full-square summary features
structured 3×3 square-grid features

If you want, next I can give you a companion cell that compares this richer-feature model directly against your best previous ExtraTrees model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

# best current mixed-size config
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

# tight square around corner ROI circles
EXTRA_MARGIN = 0

# square grid: 2 means 2x2 cells, 3 means 3x3 cells
GRID_N = 3

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_sizes, extra_margin=0):
    # use outer edges of actual corner ROI circles
    lefts = []
    rights = []
    tops = []
    bottoms = []

    for roi_name, (x, y) in corner_points_xy.items():
        r = roi_sizes[roi_name] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts) - extra_margin)
    right = int(max(rights) + extra_margin)
    top = int(min(tops) - extra_margin)
    bottom = int(max(bottoms) + extra_margin)

    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_basic_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    # simple color summary
    rgb_sum = r + g + b + 1e-8
    sat_proxy = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) / rgb_sum

    # gradient features on luma
    gy, gx = np.gradient(luma)
    grad_mag = np.sqrt(gx**2 + gy**2)

    # highlight / shadow proxies from luma percentiles inside patch
    p10 = np.percentile(luma, 10)
    p90 = np.percentile(luma, 90)
    highlight_frac = float(np.mean(luma >= p90))
    shadow_frac = float(np.mean(luma <= p10))

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
        "mean_sat_proxy": float(sat_proxy.mean()),
        "std_sat_proxy": float(sat_proxy.std()),
        "grad_mean": float(grad_mag.mean()),
        "grad_std": float(grad_mag.std()),
        "highlight_frac": highlight_frac,
        "shadow_frac": shadow_frac,
    }

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch)
    return compute_basic_features_from_array(arr)

def compute_grid_features_from_patch(patch_img, grid_n=3, prefix="sq"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)
            # keep a compact subset per cell
            keep = ["mean_luma", "std_luma", "grad_mean", "mean_sat_proxy"]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]
    return feats

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

# =========================================================
# BUILD IMAGE-LEVEL DATASET
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []

for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        rec = {
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
        }

        # store ROI coordinates
        roi_luma_means = {}
        roi_grad_means = {}

        for _, row in g.iterrows():
            roi = row["roi_name"]
            x = int(row["x"])
            y = int(row["y"])
            rec[f"x_{roi}"] = x
            rec[f"y_{roi}"] = y

            feats = compute_patch_features(img, x, y, ROI_SIZES[roi])
            for k, v in feats.items():
                rec[f"{roi}_{k}"] = v

            roi_luma_means[roi] = feats["mean_luma"]
            roi_grad_means[roi] = feats["grad_mean"]

        # relational ROI features
        corner_lumas = np.array([roi_luma_means[r] for r in CORNER_ROIS], dtype=float)
        corner_grads = np.array([roi_grad_means[r] for r in CORNER_ROIS], dtype=float)

        rec["corner_mean_luma"] = float(corner_lumas.mean())
        rec["corner_std_luma"] = float(corner_lumas.std())
        rec["corner_min_luma"] = float(corner_lumas.min())
        rec["corner_max_luma"] = float(corner_lumas.max())
        rec["corner_range_luma"] = float(corner_lumas.max() - corner_lumas.min())

        rec["C_minus_corner_mean_luma"] = float(roi_luma_means["C"] - corner_lumas.mean())
        rec["C_over_corner_mean_luma"] = float(roi_luma_means["C"] / (corner_lumas.mean() + 1e-8))

        rec["corner_mean_grad"] = float(corner_grads.mean())
        rec["corner_std_grad"] = float(corner_grads.std())
        rec["C_minus_corner_mean_grad"] = float(roi_grad_means["C"] - corner_grads.mean())

        # pairwise center-to-corner diffs
        for roi in CORNER_ROIS:
            rec[f"C_minus_{roi}_luma"] = float(roi_luma_means["C"] - roi_luma_means[roi])

        # square / bbox features
        corner_points_xy = {
            "UL": (rec["x_UL"], rec["y_UL"]),
            "UR": (rec["x_UR"], rec["y_UR"]),
            "LR": (rec["x_LR"], rec["y_LR"]),
            "LL": (rec["x_LL"], rec["y_LL"]),
        }

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, ROI_SIZES, extra_margin=EXTRA_MARGIN
        )
        square_patch = crop_bbox(img, left, top, right, bottom)

        square_basic = compute_basic_features_from_array(np.asarray(square_patch))
        for k, v in square_basic.items():
            rec[f"square_{k}"] = v

        square_grid = compute_grid_features_from_patch(square_patch, grid_n=GRID_N, prefix="square")
        rec.update(square_grid)

        # ROI vs square relationships
        rec["C_minus_square_mean_luma"] = float(rec["C_mean_luma"] - rec["square_mean_luma"])
        rec["corner_mean_minus_square_mean_luma"] = float(rec["corner_mean_luma"] - rec["square_mean_luma"])
        rec["C_over_square_mean_luma"] = float(rec["C_mean_luma"] / (rec["square_mean_luma"] + 1e-8))

        rows.append(rec)

    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))
print("Columns:", len(df.columns))

# =========================================================
# FEATURE SET
# =========================================================
exclude_cols = {
    "image_path", "filename", "session", "table_type", "surface", "target_lux"
}
exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

feature_cols = [c for c in df.columns if c not in exclude_cols]

print("\nNumber of feature columns:", len(feature_cols))
print("Sample feature columns:")
print(feature_cols[:40])

# =========================================================
# TRAIN / EVAL
# =========================================================
subsets = make_subsets(df)
all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    train_df = df_sub.iloc[train_idx].copy()
    test_df = df_sub.iloc[test_idx].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df["target_lux"].values
    X_test = test_df[feature_cols].values
    y_test = test_df["target_lux"].values

    model = ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics_row = {
        "subset": subset_name,
        "model": "ExtraTrees_richer_features",
        "n_features": len(feature_cols),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(metrics_row)
    print(pd.DataFrame([metrics_row]).to_string(index=False))

    pred_df = test_df.copy()
    pred_df["pred"] = y_pred
    pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
    pred_df["pct_err"] = np.abs(
        (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
    ) * 100.0
    pred_df.to_csv(OUT_DIR / f"{subset_name}__predictions.csv", index=False)

    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    fi_df.to_csv(OUT_DIR / f"{subset_name}__feature_importance.csv", index=False)

metrics_df = pd.DataFrame(all_metrics).sort_values("mae").reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

In [ ]:
df.to_csv(OUT_DIR / "extracted_features_master.csv", index=False)
print("Saved:", OUT_DIR / "extracted_features_master.csv")

Summary metrics plots

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

# best current mixed-size config
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

# tight square around corner ROI circles
EXTRA_MARGIN = 0

# square grid: 2 means 2x2 cells, 3 means 3x3 cells
GRID_N = 3

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_sizes, extra_margin=0):
    # use outer edges of actual corner ROI circles
    lefts = []
    rights = []
    tops = []
    bottoms = []

    for roi_name, (x, y) in corner_points_xy.items():
        r = roi_sizes[roi_name] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts) - extra_margin)
    right = int(max(rights) + extra_margin)
    top = int(min(tops) - extra_margin)
    bottom = int(max(bottoms) + extra_margin)

    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_basic_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    # simple color summary
    rgb_sum = r + g + b + 1e-8
    sat_proxy = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) / rgb_sum

    # gradient features on luma
    gy, gx = np.gradient(luma)
    grad_mag = np.sqrt(gx**2 + gy**2)

    # highlight / shadow proxies from luma percentiles inside patch
    p10 = np.percentile(luma, 10)
    p90 = np.percentile(luma, 90)
    highlight_frac = float(np.mean(luma >= p90))
    shadow_frac = float(np.mean(luma <= p10))

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
        "mean_sat_proxy": float(sat_proxy.mean()),
        "std_sat_proxy": float(sat_proxy.std()),
        "grad_mean": float(grad_mag.mean()),
        "grad_std": float(grad_mag.std()),
        "highlight_frac": highlight_frac,
        "shadow_frac": shadow_frac,
    }

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    arr = np.asarray(patch)
    return compute_basic_features_from_array(arr)

def compute_grid_features_from_patch(patch_img, grid_n=3, prefix="sq"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)
            # keep a compact subset per cell
            keep = ["mean_luma", "std_luma", "grad_mean", "mean_sat_proxy"]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]
    return feats

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

# =========================================================
# BUILD IMAGE-LEVEL DATASET
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []

for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        rec = {
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
        }

        # store ROI coordinates
        roi_luma_means = {}
        roi_grad_means = {}

        for _, row in g.iterrows():
            roi = row["roi_name"]
            x = int(row["x"])
            y = int(row["y"])
            rec[f"x_{roi}"] = x
            rec[f"y_{roi}"] = y

            feats = compute_patch_features(img, x, y, ROI_SIZES[roi])
            for k, v in feats.items():
                rec[f"{roi}_{k}"] = v

            roi_luma_means[roi] = feats["mean_luma"]
            roi_grad_means[roi] = feats["grad_mean"]

        # relational ROI features
        corner_lumas = np.array([roi_luma_means[r] for r in CORNER_ROIS], dtype=float)
        corner_grads = np.array([roi_grad_means[r] for r in CORNER_ROIS], dtype=float)

        rec["corner_mean_luma"] = float(corner_lumas.mean())
        rec["corner_std_luma"] = float(corner_lumas.std())
        rec["corner_min_luma"] = float(corner_lumas.min())
        rec["corner_max_luma"] = float(corner_lumas.max())
        rec["corner_range_luma"] = float(corner_lumas.max() - corner_lumas.min())

        rec["C_minus_corner_mean_luma"] = float(roi_luma_means["C"] - corner_lumas.mean())
        rec["C_over_corner_mean_luma"] = float(roi_luma_means["C"] / (corner_lumas.mean() + 1e-8))

        rec["corner_mean_grad"] = float(corner_grads.mean())
        rec["corner_std_grad"] = float(corner_grads.std())
        rec["C_minus_corner_mean_grad"] = float(roi_grad_means["C"] - corner_grads.mean())

        # pairwise center-to-corner diffs
        for roi in CORNER_ROIS:
            rec[f"C_minus_{roi}_luma"] = float(roi_luma_means["C"] - roi_luma_means[roi])

        # square / bbox features
        corner_points_xy = {
            "UL": (rec["x_UL"], rec["y_UL"]),
            "UR": (rec["x_UR"], rec["y_UR"]),
            "LR": (rec["x_LR"], rec["y_LR"]),
            "LL": (rec["x_LL"], rec["y_LL"]),
        }

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, ROI_SIZES, extra_margin=EXTRA_MARGIN
        )
        square_patch = crop_bbox(img, left, top, right, bottom)

        square_basic = compute_basic_features_from_array(np.asarray(square_patch))
        for k, v in square_basic.items():
            rec[f"square_{k}"] = v

        square_grid = compute_grid_features_from_patch(square_patch, grid_n=GRID_N, prefix="square")
        rec.update(square_grid)

        # ROI vs square relationships
        rec["C_minus_square_mean_luma"] = float(rec["C_mean_luma"] - rec["square_mean_luma"])
        rec["corner_mean_minus_square_mean_luma"] = float(rec["corner_mean_luma"] - rec["square_mean_luma"])
        rec["C_over_square_mean_luma"] = float(rec["C_mean_luma"] / (rec["square_mean_luma"] + 1e-8))

        rows.append(rec)

    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))
print("Columns:", len(df.columns))

# =========================================================
# FEATURE SET
# =========================================================
exclude_cols = {
    "image_path", "filename", "session", "table_type", "surface", "target_lux"
}
exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

feature_cols = [c for c in df.columns if c not in exclude_cols]

print("\nNumber of feature columns:", len(feature_cols))
print("Sample feature columns:")
print(feature_cols[:40])

# =========================================================
# TRAIN / EVAL
# =========================================================
subsets = make_subsets(df)
all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    train_df = df_sub.iloc[train_idx].copy()
    test_df = df_sub.iloc[test_idx].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df["target_lux"].values
    X_test = test_df[feature_cols].values
    y_test = test_df["target_lux"].values

    model = ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics_row = {
        "subset": subset_name,
        "model": "ExtraTrees_richer_features",
        "n_features": len(feature_cols),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(metrics_row)
    print(pd.DataFrame([metrics_row]).to_string(index=False))

    pred_df = test_df.copy()
    pred_df["pred"] = y_pred
    pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
    pred_df["pct_err"] = np.abs(
        (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
    ) * 100.0
    pred_df.to_csv(OUT_DIR / f"{subset_name}__predictions.csv", index=False)

    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    fi_df.to_csv(OUT_DIR / f"{subset_name}__feature_importance.csv", index=False)

metrics_df = pd.DataFrame(all_metrics).sort_values("mae").reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

Predicted vs true and error plots for each subset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features")
OUT_DIR = BASE_DIR / "visualizations_predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_files = sorted(BASE_DIR.glob("*__predictions.csv"))
print("Found prediction files:", len(pred_files))
for p in pred_files:
    print("-", p.name)

for p in pred_files:
    df = pd.read_csv(p)
    name = p.stem.replace("__predictions", "")

    true = df["target_lux"].values
    pred = df["pred"].values
    abs_err = df["abs_err"].values
    pct_err = df["pct_err"].values
    resid = pred - true

    # predicted vs true
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.5)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(f"Predicted vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pred_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # absolute error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, abs_err, alpha=0.5)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(f"Absolute Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # percent error vs true
    plt.figure(figsize=(6, 5))
    plt.scatter(true, pct_err, alpha=0.5)
    plt.xlabel("True lux")
    plt.ylabel("Absolute percentage error (%)")
    plt.title(f"Percentage Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # residuals
    plt.figure(figsize=(6, 5))
    plt.scatter(true, resid, alpha=0.5)
    plt.axhline(0, linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Residual (pred - true)")
    plt.title(f"Residual Plot\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__residuals.png", dpi=220, bbox_inches="tight")
    plt.show()

    # abs error hist
    plt.figure(figsize=(6, 5))
    plt.hist(abs_err, bins=30)
    plt.xlabel("Absolute error (lux)")
    plt.ylabel("Count")
    plt.title(f"Absolute Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

    # pct error hist
    plt.figure(figsize=(6, 5))
    plt.hist(pct_err, bins=30)
    plt.xlabel("Absolute percentage error (%)")
    plt.ylabel("Count")
    plt.title(f"Percentage Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved prediction plots to:", OUT_DIR)

One compact comparison figure across subsets

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features")
OUT_DIR = BASE_DIR / "visualizations_feature_importance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

fi_files = sorted(BASE_DIR.glob("*__feature_importance.csv"))
print("Found feature importance files:", len(fi_files))
for p in fi_files:
    print("-", p.name)

for p in fi_files:
    df = pd.read_csv(p)
    name = p.stem.replace("__feature_importance", "")

    top = df.head(20).iloc[::-1]

    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("Importance")
    plt.title(f"Top 20 Feature Importances\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__top20_feature_importance.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved feature-importance plots to:", OUT_DIR)

compare 3×3, 4×4, 5×5, 8×8 square grids

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_grid_size_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

# fixed ROI sizes from current best mixed-size setup
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

GRID_SIZES = [3, 4, 5, 8]
EXTRA_MARGIN = 0

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_sizes, extra_margin=0):
    lefts, rights, tops, bottoms = [], [], [], []

    for roi_name, (x, y) in corner_points_xy.items():
        r = roi_sizes[roi_name] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts) - extra_margin)
    right = int(max(rights) + extra_margin)
    top = int(min(tops) - extra_margin)
    bottom = int(max(bottoms) + extra_margin)

    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_basic_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    rgb_sum = r + g + b + 1e-8
    sat_proxy = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) / rgb_sum

    gy, gx = np.gradient(luma)
    grad_mag = np.sqrt(gx**2 + gy**2)

    p10 = np.percentile(luma, 10)
    p90 = np.percentile(luma, 90)

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
        "mean_sat_proxy": float(sat_proxy.mean()),
        "std_sat_proxy": float(sat_proxy.std()),
        "grad_mean": float(grad_mag.mean()),
        "grad_std": float(grad_mag.std()),
        "highlight_frac": float(np.mean(luma >= p90)),
        "shadow_frac": float(np.mean(luma <= p10)),
    }

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    return compute_basic_features_from_array(np.asarray(patch))

def compute_grid_features_from_patch(patch_img, grid_n=3, prefix="square"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)

            keep = ["mean_luma", "std_luma", "grad_mean", "mean_sat_proxy"]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]
    return feats

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

# =========================================================
# LOAD BASE IMAGE-LEVEL ROWS
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

base_rows = []

for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    rec = {
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
    }

    for _, row in g.iterrows():
        roi = row["roi_name"]
        rec[f"x_{roi}"] = int(row["x"])
        rec[f"y_{roi}"] = int(row["y"])

    base_rows.append(rec)

base_df = pd.DataFrame(base_rows)
print("Base image-level rows:", len(base_df))

all_metrics = []

# =========================================================
# RUN GRID-SIZE EXPERIMENTS
# =========================================================
for GRID_N in GRID_SIZES:
    print(f"\n================ GRID = {GRID_N}x{GRID_N} ================\n")

    rows = []

    for _, row in base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = row.to_dict()
            rec["grid_n"] = GRID_N

            roi_luma_means = {}
            roi_grad_means = {}

            # per-ROI features
            for roi in ROI_ORDER:
                x = row[f"x_{roi}"]
                y = row[f"y_{roi}"]
                feats = compute_patch_features(img, x, y, ROI_SIZES[roi])
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v
                roi_luma_means[roi] = feats["mean_luma"]
                roi_grad_means[roi] = feats["grad_mean"]

            # relational ROI features
            corner_lumas = np.array([roi_luma_means[r] for r in CORNER_ROIS], dtype=float)
            corner_grads = np.array([roi_grad_means[r] for r in CORNER_ROIS], dtype=float)

            rec["corner_mean_luma"] = float(corner_lumas.mean())
            rec["corner_std_luma"] = float(corner_lumas.std())
            rec["corner_min_luma"] = float(corner_lumas.min())
            rec["corner_max_luma"] = float(corner_lumas.max())
            rec["corner_range_luma"] = float(corner_lumas.max() - corner_lumas.min())

            rec["C_minus_corner_mean_luma"] = float(roi_luma_means["C"] - corner_lumas.mean())
            rec["C_over_corner_mean_luma"] = float(roi_luma_means["C"] / (corner_lumas.mean() + 1e-8))

            rec["corner_mean_grad"] = float(corner_grads.mean())
            rec["corner_std_grad"] = float(corner_grads.std())
            rec["C_minus_corner_mean_grad"] = float(roi_grad_means["C"] - corner_grads.mean())

            for roi in CORNER_ROIS:
                rec[f"C_minus_{roi}_luma"] = float(roi_luma_means["C"] - roi_luma_means[roi])

            # square features
            corner_points_xy = {
                "UL": (row["x_UL"], row["y_UL"]),
                "UR": (row["x_UR"], row["y_UR"]),
                "LR": (row["x_LR"], row["y_LR"]),
                "LL": (row["x_LL"], row["y_LL"]),
            }

            left, top, right, bottom = get_tight_bbox_from_corner_rois(
                corner_points_xy, ROI_SIZES, extra_margin=EXTRA_MARGIN
            )
            square_patch = crop_bbox(img, left, top, right, bottom)

            square_basic = compute_basic_features_from_array(np.asarray(square_patch))
            for k, v in square_basic.items():
                rec[f"square_{k}"] = v

            square_grid = compute_grid_features_from_patch(square_patch, grid_n=GRID_N, prefix="square")
            rec.update(square_grid)

            rec["C_minus_square_mean_luma"] = float(rec["C_mean_luma"] - rec["square_mean_luma"])
            rec["corner_mean_minus_square_mean_luma"] = float(rec["corner_mean_luma"] - rec["square_mean_luma"])
            rec["C_over_square_mean_luma"] = float(rec["C_mean_luma"] / (rec["square_mean_luma"] + 1e-8))

            rows.append(rec)

        except Exception as e:
            print("Skipped:", row["image_path"], e)

    df = pd.DataFrame(rows)
    print("Rows built:", len(df))

    exclude_cols = {
        "image_path", "filename", "session", "table_type", "surface", "target_lux", "grid_n"
    }
    exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
    exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print("Number of feature columns:", len(feature_cols))

    subsets = make_subsets(df)

    for subset_name, df_sub in subsets.items():
        print(f"=== {subset_name} ===")
        print("Rows:", len(df_sub))
        if len(df_sub) < 20:
            continue

        X = df_sub[feature_cols].values
        y = df_sub["target_lux"].values
        groups = df_sub["session"].astype(str).values

        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))

        train_df = df_sub.iloc[train_idx].copy()
        test_df = df_sub.iloc[test_idx].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df["target_lux"].values
        X_test = test_df[feature_cols].values
        y_test = test_df["target_lux"].values

        model = ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics_row = {
            "subset": subset_name,
            "model": "ExtraTrees_grid_compare",
            "grid_n": GRID_N,
            "n_features": len(feature_cols),
            "n_train": len(train_df),
            "n_test": len(test_df),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_metrics.append(metrics_row)
        print(pd.DataFrame([metrics_row]).to_string(index=False))

        pred_df = test_df.copy()
        pred_df["pred"] = y_pred
        pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
        pred_df["pct_err"] = np.abs(
            (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
        ) * 100.0
        pred_df.to_csv(OUT_DIR / f"grid{GRID_N}__{subset_name}__predictions.csv", index=False)

        fi_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        fi_df.to_csv(OUT_DIR / f"grid{GRID_N}__{subset_name}__feature_importance.csv", index=False)

metrics_df = pd.DataFrame(all_metrics).sort_values(["subset", "mae"]).reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics_grid_compare.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_grid_compare.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_grid_size_comparison")
df = pd.read_csv(BASE_DIR / "metrics_grid_compare.csv")

print(df.to_string(index=False))

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]
subset_label_map = {
    "white_paper_only": "White paper",
    "tables_only": "Tables",
    "white_plus_tables": "White + tables",
    "all_5roi_data": "All 5-ROI data",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, subset in zip(axes, subset_order):
    sub = df[df["subset"] == subset].sort_values("grid_n")
    ax.plot(sub["grid_n"], sub["mae"], marker="o")
    ax.set_title(subset_label_map[subset])
    ax.set_xlabel("Square grid size")
    ax.set_ylabel("MAE (lux)")
    ax.set_xticks(sub["grid_n"])

plt.tight_layout()
plt.savefig(BASE_DIR / "grid_compare__mae.png", dpi=220, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, subset in zip(axes, subset_order):
    sub = df[df["subset"] == subset].sort_values("grid_n")
    ax.plot(sub["grid_n"], sub["r2"], marker="o")
    ax.set_title(subset_label_map[subset])
    ax.set_xlabel("Square grid size")
    ax.set_ylabel("R²")
    ax.set_xticks(sub["grid_n"])

plt.tight_layout()
plt.savefig(BASE_DIR / "grid_compare__r2.png", dpi=220, bbox_inches="tight")
plt.show()

Extended features for square

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

# current best mixed ROI size setup
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

GRID_N = 5
EXTRA_MARGIN = 0

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_sizes, extra_margin=0):
    lefts, rights, tops, bottoms = [], [], [], []

    for roi_name, (x, y) in corner_points_xy.items():
        r = roi_sizes[roi_name] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts) - extra_margin)
    right = int(max(rights) + extra_margin)
    top = int(min(tops) - extra_margin)
    bottom = int(max(bottoms) + extra_margin)

    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_basic_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    rgb_sum = r + g + b + 1e-8
    sat_proxy = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) / rgb_sum

    gy, gx = np.gradient(luma)
    grad_mag = np.sqrt(gx**2 + gy**2)

    p10 = np.percentile(luma, 10)
    p50 = np.percentile(luma, 50)
    p90 = np.percentile(luma, 90)

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
        "luma_p10": float(p10),
        "luma_p50": float(p50),
        "luma_p90": float(p90),
        "mean_sat_proxy": float(sat_proxy.mean()),
        "std_sat_proxy": float(sat_proxy.std()),
        "grad_mean": float(grad_mag.mean()),
        "grad_std": float(grad_mag.std()),
        "highlight_frac": float(np.mean(luma >= p90)),
        "shadow_frac": float(np.mean(luma <= p10)),
    }

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    return compute_basic_features_from_array(np.asarray(patch))

def compute_grid_features_from_patch(patch_img, grid_n=5, prefix="square"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    cell_means = np.zeros((grid_n, grid_n), dtype=float)

    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)

            keep = [
                "mean_luma",
                "std_luma",
                "grad_mean",
                "grad_std",
                "mean_sat_proxy",
                "highlight_frac",
                "shadow_frac",
                "luma_p10",
                "luma_p50",
                "luma_p90",
            ]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]

            cell_means[i, j] = cell_feats["mean_luma"]

    # -----------------------------------------------------
    # relational summary features across the square grid
    # -----------------------------------------------------
    top_mean = float(cell_means[0, :].mean())
    bottom_mean = float(cell_means[-1, :].mean())
    left_mean = float(cell_means[:, 0].mean())
    right_mean = float(cell_means[:, -1].mean())
    center_mean = float(cell_means[grid_n // 2, grid_n // 2])

    corner_mean = float(np.mean([
        cell_means[0, 0],
        cell_means[0, -1],
        cell_means[-1, 0],
        cell_means[-1, -1],
    ]))

    feats[f"{prefix}_top_mean_luma"] = top_mean
    feats[f"{prefix}_bottom_mean_luma"] = bottom_mean
    feats[f"{prefix}_left_mean_luma"] = left_mean
    feats[f"{prefix}_right_mean_luma"] = right_mean
    feats[f"{prefix}_center_cell_mean_luma"] = center_mean
    feats[f"{prefix}_corner_cells_mean_luma"] = corner_mean

    feats[f"{prefix}_bottom_minus_top_luma"] = bottom_mean - top_mean
    feats[f"{prefix}_right_minus_left_luma"] = right_mean - left_mean
    feats[f"{prefix}_center_minus_corners_luma"] = center_mean - corner_mean

    feats[f"{prefix}_grid_max_mean_luma"] = float(cell_means.max())
    feats[f"{prefix}_grid_min_mean_luma"] = float(cell_means.min())
    feats[f"{prefix}_grid_range_mean_luma"] = float(cell_means.max() - cell_means.min())
    feats[f"{prefix}_grid_std_mean_luma"] = float(cell_means.std())

    return feats

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

# =========================================================
# BUILD IMAGE-LEVEL DATASET
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

rows = []

for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img).convert("RGB")

        rec = {
            "image_path": image_path,
            "filename": g.iloc[0]["filename"],
            "session": str(g.iloc[0]["session"]),
            "table_type": str(g.iloc[0]["table_type"]),
            "surface": str(g.iloc[0]["surface"]).lower(),
            "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
        }

        roi_luma_means = {}
        roi_grad_means = {}

        # per-ROI features
        for _, row in g.iterrows():
            roi = row["roi_name"]
            x = int(row["x"])
            y = int(row["y"])
            rec[f"x_{roi}"] = x
            rec[f"y_{roi}"] = y

            feats = compute_patch_features(img, x, y, ROI_SIZES[roi])
            for k, v in feats.items():
                rec[f"{roi}_{k}"] = v

            roi_luma_means[roi] = feats["mean_luma"]
            roi_grad_means[roi] = feats["grad_mean"]

        # relational ROI features
        corner_lumas = np.array([roi_luma_means[r] for r in CORNER_ROIS], dtype=float)
        corner_grads = np.array([roi_grad_means[r] for r in CORNER_ROIS], dtype=float)

        rec["corner_mean_luma"] = float(corner_lumas.mean())
        rec["corner_std_luma"] = float(corner_lumas.std())
        rec["corner_min_luma"] = float(corner_lumas.min())
        rec["corner_max_luma"] = float(corner_lumas.max())
        rec["corner_range_luma"] = float(corner_lumas.max() - corner_lumas.min())

        rec["C_minus_corner_mean_luma"] = float(roi_luma_means["C"] - corner_lumas.mean())
        rec["C_over_corner_mean_luma"] = float(roi_luma_means["C"] / (corner_lumas.mean() + 1e-8))

        rec["corner_mean_grad"] = float(corner_grads.mean())
        rec["corner_std_grad"] = float(corner_grads.std())
        rec["C_minus_corner_mean_grad"] = float(roi_grad_means["C"] - corner_grads.mean())

        for roi in CORNER_ROIS:
            rec[f"C_minus_{roi}_luma"] = float(roi_luma_means["C"] - roi_luma_means[roi])

        # square features
        corner_points_xy = {
            "UL": (rec["x_UL"], rec["y_UL"]),
            "UR": (rec["x_UR"], rec["y_UR"]),
            "LR": (rec["x_LR"], rec["y_LR"]),
            "LL": (rec["x_LL"], rec["y_LL"]),
        }

        left, top, right, bottom = get_tight_bbox_from_corner_rois(
            corner_points_xy, ROI_SIZES, extra_margin=EXTRA_MARGIN
        )
        square_patch = crop_bbox(img, left, top, right, bottom)

        square_basic = compute_basic_features_from_array(np.asarray(square_patch))
        for k, v in square_basic.items():
            rec[f"square_{k}"] = v

        square_grid = compute_grid_features_from_patch(square_patch, grid_n=GRID_N, prefix="square")
        rec.update(square_grid)

        rec["C_minus_square_mean_luma"] = float(rec["C_mean_luma"] - rec["square_mean_luma"])
        rec["corner_mean_minus_square_mean_luma"] = float(rec["corner_mean_luma"] - rec["square_mean_luma"])
        rec["C_over_square_mean_luma"] = float(rec["C_mean_luma"] / (rec["square_mean_luma"] + 1e-8))

        rows.append(rec)

    except Exception as e:
        print("Skipped:", image_path, e)

df = pd.DataFrame(rows)
print("Image-level rows:", len(df))
print("Columns:", len(df.columns))

# save full extracted feature table
df.to_csv(OUT_DIR / "extracted_features_master.csv", index=False)
print("Saved:", OUT_DIR / "extracted_features_master.csv")

# =========================================================
# FEATURE SET
# =========================================================
exclude_cols = {
    "image_path", "filename", "session", "table_type", "surface", "target_lux"
}
exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

feature_cols = [c for c in df.columns if c not in exclude_cols]

print("\nNumber of feature columns:", len(feature_cols))
print("Sample feature columns:")
print(feature_cols[:50])

# =========================================================
# TRAIN / EVAL
# =========================================================
subsets = make_subsets(df)
all_metrics = []

for subset_name, df_sub in subsets.items():
    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    if len(df_sub) < 20:
        continue

    X = df_sub[feature_cols].values
    y = df_sub["target_lux"].values
    groups = df_sub["session"].astype(str).values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    train_df = df_sub.iloc[train_idx].copy()
    test_df = df_sub.iloc[test_idx].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df["target_lux"].values
    X_test = test_df[feature_cols].values
    y_test = test_df["target_lux"].values

    model = ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics_row = {
        "subset": subset_name,
        "model": "ExtraTrees_richer_features_plus_square",
        "grid_n": GRID_N,
        "n_features": len(feature_cols),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred),
        "mape_percent": mape_percent(y_test, y_pred),
    }
    all_metrics.append(metrics_row)
    print(pd.DataFrame([metrics_row]).to_string(index=False))

    pred_df = test_df.copy()
    pred_df["pred"] = y_pred
    pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
    pred_df["pct_err"] = np.abs(
        (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
    ) * 100.0
    pred_df.to_csv(OUT_DIR / f"{subset_name}__predictions.csv", index=False)

    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    fi_df.to_csv(OUT_DIR / f"{subset_name}__feature_importance.csv", index=False)

metrics_df = pd.DataFrame(all_metrics).sort_values("mae").reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR = BASE_DIR / "visualizations"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_files = sorted(BASE_DIR.glob("*__predictions.csv"))
print("Found prediction files:", len(pred_files))
for p in pred_files:
    print("-", p.name)

for p in pred_files:
    df = pd.read_csv(p)
    name = p.stem.replace("__predictions", "")

    true = df["target_lux"].values
    pred = df["pred"].values
    abs_err = df["abs_err"].values
    pct_err = df["pct_err"].values
    resid = pred - true

    # Predicted vs True
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.55)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(f"Predicted vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pred_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Absolute error vs True
    plt.figure(figsize=(6, 5))
    plt.scatter(true, abs_err, alpha=0.55)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(f"Absolute Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Percent error vs True
    plt.figure(figsize=(6, 5))
    plt.scatter(true, pct_err, alpha=0.55)
    plt.xlabel("True lux")
    plt.ylabel("Absolute percentage error (%)")
    plt.title(f"Percentage Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Residuals
    plt.figure(figsize=(6, 5))
    plt.scatter(true, resid, alpha=0.55)
    plt.axhline(0, linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Residual (pred - true)")
    plt.title(f"Residual Plot\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__residuals.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Absolute error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(abs_err, bins=30)
    plt.xlabel("Absolute error (lux)")
    plt.ylabel("Count")
    plt.title(f"Absolute Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Percent error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(pct_err, bins=30)
    plt.xlabel("Absolute percentage error (%)")
    plt.ylabel("Count")
    plt.title(f"Percentage Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved plots to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR = BASE_DIR / "visualizations"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_files = sorted(BASE_DIR.glob("*__predictions.csv"))
print("Found prediction files:", len(pred_files))
for p in pred_files:
    print("-", p.name)

for p in pred_files:
    df = pd.read_csv(p)
    name = p.stem.replace("__predictions", "")

    true = df["target_lux"].values
    pred = df["pred"].values
    abs_err = df["abs_err"].values
    pct_err = df["pct_err"].values
    resid = pred - true

    # Predicted vs True
    plt.figure(figsize=(6, 6))
    plt.scatter(true, pred, alpha=0.55)
    mn = min(true.min(), pred.min())
    mx = max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Predicted lux")
    plt.title(f"Predicted vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pred_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Absolute error vs True
    plt.figure(figsize=(6, 5))
    plt.scatter(true, abs_err, alpha=0.55)
    plt.xlabel("True lux")
    plt.ylabel("Absolute error (lux)")
    plt.title(f"Absolute Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Percent error vs True
    plt.figure(figsize=(6, 5))
    plt.scatter(true, pct_err, alpha=0.55)
    plt.xlabel("True lux")
    plt.ylabel("Absolute percentage error (%)")
    plt.title(f"Percentage Error vs True\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_vs_true.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Residuals
    plt.figure(figsize=(6, 5))
    plt.scatter(true, resid, alpha=0.55)
    plt.axhline(0, linestyle="--")
    plt.xlabel("True lux")
    plt.ylabel("Residual (pred - true)")
    plt.title(f"Residual Plot\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__residuals.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Absolute error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(abs_err, bins=30)
    plt.xlabel("Absolute error (lux)")
    plt.ylabel("Count")
    plt.title(f"Absolute Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__abs_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

    # Percent error histogram
    plt.figure(figsize=(6, 5))
    plt.hist(pct_err, bins=30)
    plt.xlabel("Absolute percentage error (%)")
    plt.ylabel("Count")
    plt.title(f"Percentage Error Histogram\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__pct_err_hist.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved plots to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR = BASE_DIR / "visualizations_examples"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]

def draw_overlay(image_path):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    for roi_name, (x, y) in ROI_COORDS.items():
        r = ROI_SIZES[roi_name] // 2
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")
    return img

for subset in subset_order:
    p = BASE_DIR / f"{subset}__predictions.csv"
    if not p.exists():
        continue

    df = pd.read_csv(p).sort_values("abs_err").reset_index(drop=True)
    picks = {
        "BEST": df.iloc[0],
        "MEDIAN": df.iloc[len(df)//2],
        "WORST": df.iloc[-1],
    }

    for label, row in picks.items():
        img = draw_overlay(row["image_path"])

        plt.figure(figsize=(7, 7))
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"{subset} | {label}\n"
            f"true={row['target_lux']:.1f}, pred={row['pred']:.1f}, "
            f"abs_err={row['abs_err']:.1f}, pct_err={row['pct_err']:.2f}%\n"
            f"{Path(row['image_path']).name}",
            fontsize=10
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{subset}__{label}.png", dpi=220, bbox_inches="tight")
        plt.show()

print("Saved example images to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR = BASE_DIR / "visualizations_examples"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_COORDS = {
    "C":  (1520, 2027),
    "UL": (1020, 1372),
    "UR": (2043, 1372),
    "LR": (2030, 2722),
    "LL": (1020, 2722),
}

ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]

def draw_overlay(image_path):
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    draw = ImageDraw.Draw(img)

    for roi_name, (x, y) in ROI_COORDS.items():
        r = ROI_SIZES[roi_name] // 2
        draw.ellipse([x-r, y-r, x+r, y+r], outline="red", width=4)
        draw.text((x + r + 6, y - 8), roi_name, fill="red")
    return img

for subset in subset_order:
    p = BASE_DIR / f"{subset}__predictions.csv"
    if not p.exists():
        continue

    df = pd.read_csv(p).sort_values("abs_err").reset_index(drop=True)
    picks = {
        "BEST": df.iloc[0],
        "MEDIAN": df.iloc[len(df)//2],
        "WORST": df.iloc[-1],
    }

    for label, row in picks.items():
        img = draw_overlay(row["image_path"])

        plt.figure(figsize=(7, 7))
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"{subset} | {label}\n"
            f"true={row['target_lux']:.1f}, pred={row['pred']:.1f}, "
            f"abs_err={row['abs_err']:.1f}, pct_err={row['pct_err']:.2f}%\n"
            f"{Path(row['image_path']).name}",
            fontsize=10
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{subset}__{label}.png", dpi=220, bbox_inches="tight")
        plt.show()

print("Saved example images to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square")
OUT_DIR = BASE_DIR / "visualizations_feature_importance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

fi_files = sorted(BASE_DIR.glob("*__feature_importance.csv"))
print("Found feature importance files:", len(fi_files))

for p in fi_files:
    df = pd.read_csv(p)
    name = p.stem.replace("__feature_importance", "")
    top = df.head(20).iloc[::-1]

    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("Importance")
    plt.title(f"Top 20 Feature Importances\n{name}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}__top20.png", dpi=220, bbox_inches="tight")
    plt.show()

print("Saved to:", OUT_DIR)

CNN + Square
This model:

- takes 5 ROI patches as image input
- uses a shared CNN encoder for all 5 ROI
- takes a tabular feature vector
- fuses both branches
- predicts central lux

In [ ]:
# =========================================================
# OPTION A
# 5 ROI image patches + tabular feature branch
# Predict central lux
# =========================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionA_roi5_plus_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 16
NUM_EPOCHS = 25
LR = 1e-3
WEIGHT_DECAY = 1e-4
IMG_SIZE = 96   # all ROI patches resized to same input size for CNN

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

TARGET_COL = "target_lux"
GROUP_COL = "session"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------
# DATAFRAME
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))

# start with a smaller, safer tabular feature set
# you can expand later
tabular_feature_cols = [
    # ROI luminance / contrast
    "C_mean_luma", "C_std_luma", "C_grad_mean", "C_grad_std",
    "UL_mean_luma", "UR_mean_luma", "LR_mean_luma", "LL_mean_luma",
    "UL_grad_mean", "UR_grad_mean", "LR_grad_mean", "LL_grad_mean",

    # ROI relationships
    "corner_mean_luma", "corner_std_luma", "corner_range_luma",
    "C_minus_corner_mean_luma", "C_over_corner_mean_luma",
    "C_minus_UL_luma", "C_minus_UR_luma", "C_minus_LR_luma", "C_minus_LL_luma",

    # square whole-summary
    "square_mean_luma", "square_std_luma", "square_grad_mean", "square_grad_std",
    "square_mean_sat_proxy",

    # square-grid strongest-style summaries
    "square_top_mean_luma", "square_bottom_mean_luma",
    "square_left_mean_luma", "square_right_mean_luma",
    "square_center_cell_mean_luma", "square_corner_cells_mean_luma",
    "square_bottom_minus_top_luma", "square_right_minus_left_luma",
    "square_center_minus_corners_luma",
    "square_grid_range_mean_luma", "square_grid_std_mean_luma",
]

tabular_feature_cols = [c for c in tabular_feature_cols if c in df.columns]
print("Tabular features used:", len(tabular_feature_cols))

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def pil_to_tensor(img):
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    return torch.tensor(arr, dtype=torch.float32)

def resize_pil(img, size=96):
    return img.resize((size, size))

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }

# ---------------------------------------------------------
# SPLIT
# ---------------------------------------------------------
X_dummy = np.zeros((len(df), 1))
y = df[TARGET_COL].values
groups = df[GROUP_COL].astype(str).values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X_dummy, y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

# ---------------------------------------------------------
# SCALE TABULAR FEATURES
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_tab = scaler.fit_transform(train_df[tabular_feature_cols].values)
X_test_tab = scaler.transform(test_df[tabular_feature_cols].values)

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()

for i, col in enumerate(tabular_feature_cols):
    train_df_scaled[col] = X_train_tab[:, i]
    test_df_scaled[col] = X_test_tab[:, i]

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
class HybridROI5Dataset(Dataset):
    def __init__(self, df, tabular_cols, roi_order, roi_sizes, img_size=96):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.roi_order = roi_order
        self.roi_sizes = roi_sizes
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        roi_tensors = []
        for roi in self.roi_order:
            x = int(row[f"x_{roi}"])
            y = int(row[f"y_{roi}"])
            crop_size = int(self.roi_sizes[roi])

            patch = crop_patch(img, x, y, crop_size)
            patch = resize_pil(patch, self.img_size)
            roi_tensors.append(pil_to_tensor(patch))

        roi_stack = torch.stack(roi_tensors, dim=0)  # [5, 3, H, W]

        tab = torch.tensor(row[self.tabular_cols].values.astype(np.float32), dtype=torch.float32)
        y = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)

        return {
            "roi_imgs": roi_stack,
            "tabular": tab,
            "target": y,
            "image_path": row["image_path"],
            "filename": row["filename"],
        }

train_ds = HybridROI5Dataset(train_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES, IMG_SIZE)
test_ds = HybridROI5Dataset(test_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES, IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------
class SharedROIEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class HybridROI5Model(nn.Module):
    def __init__(self, n_tabular_features, roi_embed_dim=128, tab_embed_dim=64):
        super().__init__()
        self.roi_encoder = SharedROIEncoder(out_dim=roi_embed_dim)
        self.tab_branch = TabularMLP(n_tabular_features, hidden_dim=128, out_dim=tab_embed_dim)

        fusion_dim = 5 * roi_embed_dim + tab_embed_dim
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, roi_imgs, tabular):
        # roi_imgs: [B, 5, 3, H, W]
        B, N, C, H, W = roi_imgs.shape
        roi_imgs = roi_imgs.view(B * N, C, H, W)
        roi_emb = self.roi_encoder(roi_imgs)       # [B*N, D]
        roi_emb = roi_emb.view(B, N, -1).flatten(1)  # [B, 5*D]

        tab_emb = self.tab_branch(tabular)
        fused = torch.cat([roi_emb, tab_emb], dim=1)
        out = self.head(fused).squeeze(1)
        return out

model = HybridROI5Model(n_tabular_features=len(tabular_feature_cols)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(roi_imgs, tabular)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # eval
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            roi_imgs = batch["roi_imgs"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE)
            target = batch["target"].cpu().numpy()

            pred = model(roi_imgs, tabular).cpu().numpy()

            y_true.extend(target.tolist())
            y_pred.extend(pred.tolist())

    metrics = evaluate_regression(np.array(y_true), np.array(y_pred))
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"MAE={row['mae']:.3f} | RMSE={row['rmse']:.3f} | "
        f"R2={row['r2']:.4f} | MAPE={row['mape_percent']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

# ---------------------------------------------------------
# FINAL PREDICTIONS
# ---------------------------------------------------------
model.eval()
final_rows = []

with torch.no_grad():
    for batch in test_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        preds = model(roi_imgs, tabular).cpu().numpy()

        for i in range(len(preds)):
            true_val = float(batch["target"][i].item())
            pred_val = float(preds[i])
            final_rows.append({
                "image_path": batch["image_path"][i],
                "filename": batch["filename"][i],
                "target_lux": true_val,
                "pred": pred_val,
                "abs_err": abs(pred_val - true_val),
                "pct_err": abs(pred_val - true_val) / max(abs(true_val), 1e-8) * 100.0,
            })

pred_df = pd.DataFrame(final_rows)
pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)

final_metrics = evaluate_regression(pred_df["target_lux"].values, pred_df["pred"].values)
metrics_df = pd.DataFrame([{
    "model": "Hybrid_OptionA_ROI5_plus_features",
    "n_tabular_features": len(tabular_feature_cols),
    "img_size": IMG_SIZE,
    **final_metrics
}])
metrics_df.to_csv(OUT_DIR / "final_metrics.csv", index=False)

print("\nFINAL METRICS")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionA_roi5_plus_features")

hist = pd.read_csv(BASE_DIR / "training_history.csv")
pred = pd.read_csv(BASE_DIR / "test_predictions.csv")

# training curves
plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Train loss")
plt.title("Option A training loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["mae"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Test MAE")
plt.title("Option A test MAE by epoch")
plt.tight_layout()
plt.show()

# predicted vs true
true = pred["target_lux"].values
predv = pred["pred"].values

plt.figure(figsize=(6,6))
plt.scatter(true, predv, alpha=0.5)
mn = min(true.min(), predv.min())
mx = max(true.max(), predv.max())
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("True lux")
plt.ylabel("Predicted lux")
plt.title("Option A — Predicted vs True")
plt.tight_layout()
plt.show()

# error plots
plt.figure(figsize=(6,4))
plt.hist(pred["abs_err"], bins=30)
plt.xlabel("Absolute error (lux)")
plt.ylabel("Count")
plt.title("Option A — Absolute error histogram")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(pred["target_lux"], pred["abs_err"], alpha=0.5)
plt.xlabel("True lux")
plt.ylabel("Absolute error (lux)")
plt.title("Option A — Absolute error vs True")
plt.tight_layout()
plt.show()

Square image patch + 5 ROI image patches + tabular feature branch

This model has:

- a shared CNN encoder for the 5 ROI patches
- a separate CNN encoder for the square patch
- an MLP for tabular features
- fusion of all three branches
- final lux regression head

In [ ]:
# =========================================================
# OPTION B
# Square image patch + 5 ROI image patches + tabular feature branch
# Predict central lux
# =========================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionB_square_roi5_plus_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 12
NUM_EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4

ROI_IMG_SIZE = 96
SQUARE_IMG_SIZE = 160

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

TARGET_COL = "target_lux"
GROUP_COL = "session"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------
# DATAFRAME
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))

# modest tabular feature set to start
tabular_feature_cols = [
    # ROI luminance / contrast / gradients
    "C_mean_luma", "C_std_luma", "C_grad_mean", "C_grad_std",
    "UL_mean_luma", "UR_mean_luma", "LR_mean_luma", "LL_mean_luma",
    "UL_grad_mean", "UR_grad_mean", "LR_grad_mean", "LL_grad_mean",

    # ROI relationships
    "corner_mean_luma", "corner_std_luma", "corner_range_luma",
    "C_minus_corner_mean_luma", "C_over_corner_mean_luma",
    "C_minus_UL_luma", "C_minus_UR_luma", "C_minus_LR_luma", "C_minus_LL_luma",

    # whole square
    "square_mean_luma", "square_std_luma", "square_grad_mean", "square_grad_std",
    "square_mean_sat_proxy",

    # square summaries
    "square_top_mean_luma", "square_bottom_mean_luma",
    "square_left_mean_luma", "square_right_mean_luma",
    "square_center_cell_mean_luma", "square_corner_cells_mean_luma",
    "square_bottom_minus_top_luma", "square_right_minus_left_luma",
    "square_center_minus_corners_luma",
    "square_grid_range_mean_luma", "square_grid_std_mean_luma",
]

tabular_feature_cols = [c for c in tabular_feature_cols if c in df.columns]
print("Tabular features used:", len(tabular_feature_cols))

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_square_bbox_from_row(row):
    # Use the exact same bbox logic as the feature extraction:
    # outer edges of actual corner ROI circles
    lefts, rights, tops, bottoms = [], [], [], []

    for roi in ["UL", "UR", "LR", "LL"]:
        x = int(row[f"x_{roi}"])
        y = int(row[f"y_{roi}"])
        r = ROI_SIZES[roi] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts))
    right = int(max(rights))
    top = int(min(tops))
    bottom = int(max(bottoms))

    return left, top, right, bottom

def pil_to_tensor(img):
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.tensor(arr, dtype=torch.float32)

def resize_pil(img, size):
    return img.resize((size, size))

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }

# ---------------------------------------------------------
# SPLIT
# ---------------------------------------------------------
X_dummy = np.zeros((len(df), 1))
y = df[TARGET_COL].values
groups = df[GROUP_COL].astype(str).values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X_dummy, y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

# ---------------------------------------------------------
# SCALE TABULAR FEATURES
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_tab = scaler.fit_transform(train_df[tabular_feature_cols].values)
X_test_tab = scaler.transform(test_df[tabular_feature_cols].values)

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()

for i, col in enumerate(tabular_feature_cols):
    train_df_scaled[col] = X_train_tab[:, i]
    test_df_scaled[col] = X_test_tab[:, i]

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
class HybridSquareROI5Dataset(Dataset):
    def __init__(self, df, tabular_cols, roi_order, roi_sizes,
                 roi_img_size=96, square_img_size=160):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.roi_order = roi_order
        self.roi_sizes = roi_sizes
        self.roi_img_size = roi_img_size
        self.square_img_size = square_img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        # 5 ROI patches
        roi_tensors = []
        for roi in self.roi_order:
            x = int(row[f"x_{roi}"])
            y = int(row[f"y_{roi}"])
            crop_size = int(self.roi_sizes[roi])

            patch = crop_patch(img, x, y, crop_size)
            patch = resize_pil(patch, self.roi_img_size)
            roi_tensors.append(pil_to_tensor(patch))

        roi_stack = torch.stack(roi_tensors, dim=0)  # [5, 3, H, W]

        # square patch
        left, top, right, bottom = get_square_bbox_from_row(row)
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_patch = resize_pil(square_patch, self.square_img_size)
        square_tensor = pil_to_tensor(square_patch)

        tab = torch.tensor(row[self.tabular_cols].values.astype(np.float32), dtype=torch.float32)
        y = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)

        return {
            "roi_imgs": roi_stack,
            "square_img": square_tensor,
            "tabular": tab,
            "target": y,
            "image_path": row["image_path"],
            "filename": row["filename"],
        }

train_ds = HybridSquareROI5Dataset(
    train_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)
test_ds = HybridSquareROI5Dataset(
    test_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------
class ROIEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

class SquareEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class HybridOptionBModel(nn.Module):
    def __init__(self, n_tabular_features, roi_embed_dim=128, square_embed_dim=128, tab_embed_dim=64):
        super().__init__()
        self.roi_encoder = ROIEncoder(out_dim=roi_embed_dim)        # shared across 5 ROI
        self.square_encoder = SquareEncoder(out_dim=square_embed_dim)
        self.tab_branch = TabularMLP(n_tabular_features, hidden_dim=128, out_dim=tab_embed_dim)

        fusion_dim = 5 * roi_embed_dim + square_embed_dim + tab_embed_dim

        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, roi_imgs, square_img, tabular):
        # roi_imgs: [B, 5, 3, H, W]
        B, N, C, H, W = roi_imgs.shape
        roi_imgs = roi_imgs.view(B * N, C, H, W)
        roi_emb = self.roi_encoder(roi_imgs)
        roi_emb = roi_emb.view(B, N, -1).flatten(1)

        square_emb = self.square_encoder(square_img)
        tab_emb = self.tab_branch(tabular)

        fused = torch.cat([roi_emb, square_emb, tab_emb], dim=1)
        out = self.head(fused).squeeze(1)
        return out

model = HybridOptionBModel(n_tabular_features=len(tabular_feature_cols)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(roi_imgs, square_img, tabular)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            roi_imgs = batch["roi_imgs"].to(DEVICE)
            square_img = batch["square_img"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE)
            target = batch["target"].cpu().numpy()

            pred = model(roi_imgs, square_img, tabular).cpu().numpy()

            y_true.extend(target.tolist())
            y_pred.extend(pred.tolist())

    metrics = evaluate_regression(np.array(y_true), np.array(y_pred))
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"MAE={row['mae']:.3f} | RMSE={row['rmse']:.3f} | "
        f"R2={row['r2']:.4f} | MAPE={row['mape_percent']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

# ---------------------------------------------------------
# FINAL PREDICTIONS
# ---------------------------------------------------------
model.eval()
final_rows = []

with torch.no_grad():
    for batch in test_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)

        preds = model(roi_imgs, square_img, tabular).cpu().numpy()

        for i in range(len(preds)):
            true_val = float(batch["target"][i].item())
            pred_val = float(preds[i])
            final_rows.append({
                "image_path": batch["image_path"][i],
                "filename": batch["filename"][i],
                "target_lux": true_val,
                "pred": pred_val,
                "abs_err": abs(pred_val - true_val),
                "pct_err": abs(pred_val - true_val) / max(abs(true_val), 1e-8) * 100.0,
            })

pred_df = pd.DataFrame(final_rows)
pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)

final_metrics = evaluate_regression(pred_df["target_lux"].values, pred_df["pred"].values)
metrics_df = pd.DataFrame([{
    "model": "Hybrid_OptionB_Square_ROI5_plus_features",
    "n_tabular_features": len(tabular_feature_cols),
    "roi_img_size": ROI_IMG_SIZE,
    "square_img_size": SQUARE_IMG_SIZE,
    **final_metrics
}])
metrics_df.to_csv(OUT_DIR / "final_metrics.csv", index=False)

print("\nFINAL METRICS")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
df = pd.read_csv(CSV_PATH)

subsets = {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    "Tbrown_table_only": df[(df["surface"] == "table") & (df["table_type"] == "Tbrown")].copy(),
}

for name, d in subsets.items():
    print(f"\n{name}")
    print("rows:", len(d))
    print("unique sessions:", d["session"].astype(str).nunique())
    print("unique table types:", d["table_type"].nunique())
    if len(d) > 0:
        print(d["surface"].value_counts(dropna=False))

In [ ]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
df = pd.read_csv(CSV_PATH)

subsets = {
    "all_5roi_data": df.copy(),
    "white_paper_only": df[df["surface"] == "white_paper"].copy(),
    "tables_only": df[df["surface"] == "table"].copy(),
    "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    "Tbrown_table_only": df[(df["surface"] == "table") & (df["table_type"] == "Tbrown")].copy(),
}

for name, d in subsets.items():
    print(f"\n{name}")
    print("rows:", len(d))
    print("unique sessions:", d["session"].astype(str).nunique())
    print("unique table types:", d["table_type"].nunique())
    if len(d) > 0:
        print(d["surface"].value_counts(dropna=False))

In [ ]:
# =========================================================
# OPTION B SUBSET RUNNER — FULL VERSION
# Square image patch + 5 ROI image patches + tabular feature branch
# Predict central lux
#
# Supported subset modes:
#   "all_5roi_data"
#   "white_paper_only"
#   "tables_only"
#   "white_plus_tables"
#   "Tbrown_table_only"
#   "Tbrown_white_plus_table"
#
# Notes:
# - Uses best checkpoint by validation MAE
# - Uses case-insensitive table_type filtering
# - Prints available table_type values before filtering
# =========================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
ROOT_OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionB_subset_runs")
ROOT_OUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_MODE = "white_paper_only"
# Choose one:
# "all_5roi_data"
# "white_paper_only"
# "tables_only"
# "white_plus_tables"
# "Tbrown_table_only"
# "Tbrown_white_plus_table"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 12
NUM_EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 1e-4

ROI_IMG_SIZE = 96
SQUARE_IMG_SIZE = 160

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

TARGET_COL = "target_lux"
GROUP_COL = "session"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_DIR = ROOT_OUT_DIR / SUBSET_MODE
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)
print("All rows:", len(df))
print("All columns:", len(df.columns))

print("\nUnique table_type values:")
for x in sorted(df["table_type"].dropna().astype(str).unique()):
    print(repr(x))

print("\nCounts by table_type:")
print(df["table_type"].astype(str).value_counts())

# ---------------------------------------------------------
# TABULAR FEATURES
# ---------------------------------------------------------
tabular_feature_cols = [
    # ROI luminance / contrast / gradients
    "C_mean_luma", "C_std_luma", "C_grad_mean", "C_grad_std",
    "UL_mean_luma", "UR_mean_luma", "LR_mean_luma", "LL_mean_luma",
    "UL_grad_mean", "UR_grad_mean", "LR_grad_mean", "LL_grad_mean",

    # ROI relationships
    "corner_mean_luma", "corner_std_luma", "corner_range_luma",
    "C_minus_corner_mean_luma", "C_over_corner_mean_luma",
    "C_minus_UL_luma", "C_minus_UR_luma", "C_minus_LR_luma", "C_minus_LL_luma",

    # whole square
    "square_mean_luma", "square_std_luma", "square_grad_mean", "square_grad_std",
    "square_mean_sat_proxy",

    # square summaries
    "square_top_mean_luma", "square_bottom_mean_luma",
    "square_left_mean_luma", "square_right_mean_luma",
    "square_center_cell_mean_luma", "square_corner_cells_mean_luma",
    "square_bottom_minus_top_luma", "square_right_minus_left_luma",
    "square_center_minus_corners_luma",
    "square_grid_range_mean_luma", "square_grid_std_mean_luma",
]
tabular_feature_cols = [c for c in tabular_feature_cols if c in df.columns]
print("\nTabular features used:", len(tabular_feature_cols))

# ---------------------------------------------------------
# SUBSET FILTER
# ---------------------------------------------------------
def apply_subset(df_in, subset_mode):
    df_sub = df_in.copy()

    table_type_lower = df_sub["table_type"].astype(str).str.strip().str.lower()
    surface_lower = df_sub["surface"].astype(str).str.strip().str.lower()

    if subset_mode == "all_5roi_data":
        pass

    elif subset_mode == "white_paper_only":
        df_sub = df_sub[surface_lower == "white_paper"].copy()

    elif subset_mode == "tables_only":
        df_sub = df_sub[surface_lower == "table"].copy()

    elif subset_mode == "white_plus_tables":
        df_sub = df_sub[surface_lower.isin(["white_paper", "table"])].copy()

    elif subset_mode == "Tbrown_table_only":
        brown_mask = table_type_lower == "tbrown"
        df_sub = df_sub[brown_mask & (surface_lower == "table")].copy()

    elif subset_mode == "Tbrown_white_plus_table":
        brown_mask = table_type_lower == "tbrown"
        df_sub = df_sub[brown_mask & surface_lower.isin(["white_paper", "table"])].copy()

    else:
        raise ValueError(f"Unknown SUBSET_MODE: {subset_mode}")

    return df_sub.reset_index(drop=True)

df = apply_subset(df, SUBSET_MODE)

print(f"\nSubset '{SUBSET_MODE}' rows:", len(df))
print("Unique sessions:", df[GROUP_COL].astype(str).nunique())
print("Unique table types:", df["table_type"].nunique())
if len(df) > 0:
    print(df["surface"].astype(str).value_counts(dropna=False))

if len(df) < 12:
    raise ValueError("Subset too small to train reliably.")

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_square_bbox_from_row(row):
    lefts, rights, tops, bottoms = [], [], [], []

    for roi in ["UL", "UR", "LR", "LL"]:
        x = int(row[f"x_{roi}"])
        y = int(row[f"y_{roi}"])
        r = ROI_SIZES[roi] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts))
    right = int(max(rights))
    top = int(min(tops))
    bottom = int(max(bottoms))
    return left, top, right, bottom

def pil_to_tensor(img):
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.tensor(arr, dtype=torch.float32)

def resize_pil(img, size):
    return img.resize((size, size))

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }

# ---------------------------------------------------------
# SPLIT
# ---------------------------------------------------------
X_dummy = np.zeros((len(df), 1))
y = df[TARGET_COL].values
groups = df[GROUP_COL].astype(str).values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X_dummy, y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

if len(train_df) < 10 or len(test_df) < 3:
    raise ValueError("Train/test split too small for this subset.")

# ---------------------------------------------------------
# SCALE TABULAR FEATURES
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_tab = scaler.fit_transform(train_df[tabular_feature_cols].values)
X_test_tab = scaler.transform(test_df[tabular_feature_cols].values)

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()

for i, col in enumerate(tabular_feature_cols):
    train_df_scaled[col] = X_train_tab[:, i]
    test_df_scaled[col] = X_test_tab[:, i]

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
class HybridSquareROI5Dataset(Dataset):
    def __init__(self, df, tabular_cols, roi_order, roi_sizes,
                 roi_img_size=96, square_img_size=160):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.roi_order = roi_order
        self.roi_sizes = roi_sizes
        self.roi_img_size = roi_img_size
        self.square_img_size = square_img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        roi_tensors = []
        for roi in self.roi_order:
            x = int(row[f"x_{roi}"])
            y = int(row[f"y_{roi}"])
            crop_size = int(self.roi_sizes[roi])

            patch = crop_patch(img, x, y, crop_size)
            patch = resize_pil(patch, self.roi_img_size)
            roi_tensors.append(pil_to_tensor(patch))

        roi_stack = torch.stack(roi_tensors, dim=0)

        left, top, right, bottom = get_square_bbox_from_row(row)
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_patch = resize_pil(square_patch, self.square_img_size)
        square_tensor = pil_to_tensor(square_patch)

        tab = torch.tensor(row[self.tabular_cols].values.astype(np.float32), dtype=torch.float32)
        target = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)

        return {
            "roi_imgs": roi_stack,
            "square_img": square_tensor,
            "tabular": tab,
            "target": target,
            "image_path": row["image_path"],
            "filename": row["filename"],
            "table_type": row["table_type"],
            "surface": row["surface"],
        }

train_ds = HybridSquareROI5Dataset(
    train_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)
test_ds = HybridSquareROI5Dataset(
    test_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------
class ROIEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class SquareEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class HybridOptionBModel(nn.Module):
    def __init__(self, n_tabular_features, roi_embed_dim=128, square_embed_dim=128, tab_embed_dim=64):
        super().__init__()
        self.roi_encoder = ROIEncoder(out_dim=roi_embed_dim)
        self.square_encoder = SquareEncoder(out_dim=square_embed_dim)
        self.tab_branch = TabularMLP(n_tabular_features, hidden_dim=128, out_dim=tab_embed_dim)

        fusion_dim = 5 * roi_embed_dim + square_embed_dim + tab_embed_dim
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, roi_imgs, square_img, tabular):
        B, N, C, H, W = roi_imgs.shape
        roi_imgs = roi_imgs.view(B * N, C, H, W)
        roi_emb = self.roi_encoder(roi_imgs)
        roi_emb = roi_emb.view(B, N, -1).flatten(1)

        square_emb = self.square_encoder(square_img)
        tab_emb = self.tab_branch(tabular)

        fused = torch.cat([roi_emb, square_emb, tab_emb], dim=1)
        return self.head(fused).squeeze(1)

model = HybridOptionBModel(n_tabular_features=len(tabular_feature_cols)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# ---------------------------------------------------------
# TRAIN WITH BEST CHECKPOINT BY VAL MAE
# ---------------------------------------------------------
history = []
best_mae = float("inf")
best_epoch = -1
best_state_path = OUT_DIR / "best_model.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(roi_imgs, square_img, tabular)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            roi_imgs = batch["roi_imgs"].to(DEVICE)
            square_img = batch["square_img"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE)

            preds = model(roi_imgs, square_img, tabular).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(batch["target"].numpy().tolist())

    metrics = evaluate_regression(np.array(y_true), np.array(y_pred))
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **metrics
    }
    history.append(row)

    if metrics["mae"] < best_mae:
        best_mae = metrics["mae"]
        best_epoch = epoch
        torch.save(model.state_dict(), best_state_path)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"MAE={row['mae']:.3f} | RMSE={row['rmse']:.3f} | "
        f"R2={row['r2']:.4f} | MAPE={row['mape_percent']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

print(f"\nBest epoch by MAE: {best_epoch} | best MAE={best_mae:.4f}")

# ---------------------------------------------------------
# LOAD BEST MODEL AND EVALUATE
# ---------------------------------------------------------
model.load_state_dict(torch.load(best_state_path, map_location=DEVICE))
model.eval()

final_rows = []

with torch.no_grad():
    for batch in test_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)

        preds = model(roi_imgs, square_img, tabular).cpu().numpy()

        for i in range(len(preds)):
            true_val = float(batch["target"][i].item())
            pred_val = float(preds[i])
            final_rows.append({
                "image_path": batch["image_path"][i],
                "filename": batch["filename"][i],
                "table_type": batch["table_type"][i],
                "surface": batch["surface"][i],
                "target_lux": true_val,
                "pred": pred_val,
                "abs_err": abs(pred_val - true_val),
                "pct_err": abs(pred_val - true_val) / max(abs(true_val), 1e-8) * 100.0,
            })

pred_df = pd.DataFrame(final_rows)
pred_df.to_csv(OUT_DIR / "test_predictions_best_epoch.csv", index=False)

final_metrics = evaluate_regression(pred_df["target_lux"].values, pred_df["pred"].values)
metrics_df = pd.DataFrame([{
    "subset_mode": SUBSET_MODE,
    "model": "Hybrid_OptionB_Square_ROI5_plus_features",
    "best_epoch": best_epoch,
    "n_tabular_features": len(tabular_feature_cols),
    "roi_img_size": ROI_IMG_SIZE,
    "square_img_size": SQUARE_IMG_SIZE,
    **final_metrics
}])
metrics_df.to_csv(OUT_DIR / "final_metrics_best_epoch.csv", index=False)

print("\nFINAL METRICS (BEST EPOCH)")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionB_subset_runs") / "white_paper_only"
# change the folder above to match the subset you ran

hist = pd.read_csv(BASE_DIR / "training_history.csv")
pred = pd.read_csv(BASE_DIR / "test_predictions_best_epoch.csv")

plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["mae"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("Option B — validation MAE by epoch")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,6))
true = pred["target_lux"].values
predv = pred["pred"].values
mn = min(true.min(), predv.min())
mx = max(true.max(), predv.max())
plt.scatter(true, predv, alpha=0.55)
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("True lux")
plt.ylabel("Predicted lux")
plt.title("Option B — Predicted vs True")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.hist(pred["abs_err"], bins=25)
plt.xlabel("Absolute error (lux)")
plt.ylabel("Count")
plt.title("Option B — absolute error histogram")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# OPTION B SUBSET RUNNER — FULL VERSION
# Square image patch + 5 ROI image patches + tabular feature branch
# Predict central lux
#
# Supported subset modes:
#   "all_5roi_data"
#   "white_paper_only"
#   "tables_only"
#   "white_plus_tables"
#   "Tbrown_table_only"
#   "Tbrown_white_plus_table"
#
# Notes:
# - Uses best checkpoint by validation MAE
# - Uses case-insensitive table_type filtering
# - Prints available table_type values before filtering
# =========================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
ROOT_OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionB_subset_runs")
ROOT_OUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_MODE = "tables_only"
# Choose one:
# "all_5roi_data"
# "white_paper_only"
# "tables_only"
# "white_plus_tables"
# "Tbrown_table_only"
# "Tbrown_white_plus_table"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 12
NUM_EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 1e-4

ROI_IMG_SIZE = 96
SQUARE_IMG_SIZE = 160

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

TARGET_COL = "target_lux"
GROUP_COL = "session"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_DIR = ROOT_OUT_DIR / SUBSET_MODE
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)
print("All rows:", len(df))
print("All columns:", len(df.columns))

print("\nUnique table_type values:")
for x in sorted(df["table_type"].dropna().astype(str).unique()):
    print(repr(x))

print("\nCounts by table_type:")
print(df["table_type"].astype(str).value_counts())

# ---------------------------------------------------------
# TABULAR FEATURES
# ---------------------------------------------------------
tabular_feature_cols = [
    # ROI luminance / contrast / gradients
    "C_mean_luma", "C_std_luma", "C_grad_mean", "C_grad_std",
    "UL_mean_luma", "UR_mean_luma", "LR_mean_luma", "LL_mean_luma",
    "UL_grad_mean", "UR_grad_mean", "LR_grad_mean", "LL_grad_mean",

    # ROI relationships
    "corner_mean_luma", "corner_std_luma", "corner_range_luma",
    "C_minus_corner_mean_luma", "C_over_corner_mean_luma",
    "C_minus_UL_luma", "C_minus_UR_luma", "C_minus_LR_luma", "C_minus_LL_luma",

    # whole square
    "square_mean_luma", "square_std_luma", "square_grad_mean", "square_grad_std",
    "square_mean_sat_proxy",

    # square summaries
    "square_top_mean_luma", "square_bottom_mean_luma",
    "square_left_mean_luma", "square_right_mean_luma",
    "square_center_cell_mean_luma", "square_corner_cells_mean_luma",
    "square_bottom_minus_top_luma", "square_right_minus_left_luma",
    "square_center_minus_corners_luma",
    "square_grid_range_mean_luma", "square_grid_std_mean_luma",
]
tabular_feature_cols = [c for c in tabular_feature_cols if c in df.columns]
print("\nTabular features used:", len(tabular_feature_cols))

# ---------------------------------------------------------
# SUBSET FILTER
# ---------------------------------------------------------
def apply_subset(df_in, subset_mode):
    df_sub = df_in.copy()

    table_type_lower = df_sub["table_type"].astype(str).str.strip().str.lower()
    surface_lower = df_sub["surface"].astype(str).str.strip().str.lower()

    if subset_mode == "all_5roi_data":
        pass

    elif subset_mode == "white_paper_only":
        df_sub = df_sub[surface_lower == "white_paper"].copy()

    elif subset_mode == "tables_only":
        df_sub = df_sub[surface_lower == "table"].copy()

    elif subset_mode == "white_plus_tables":
        df_sub = df_sub[surface_lower.isin(["white_paper", "table"])].copy()

    elif subset_mode == "Tbrown_table_only":
        brown_mask = table_type_lower == "tbrown"
        df_sub = df_sub[brown_mask & (surface_lower == "table")].copy()

    elif subset_mode == "Tbrown_white_plus_table":
        brown_mask = table_type_lower == "tbrown"
        df_sub = df_sub[brown_mask & surface_lower.isin(["white_paper", "table"])].copy()

    else:
        raise ValueError(f"Unknown SUBSET_MODE: {subset_mode}")

    return df_sub.reset_index(drop=True)

df = apply_subset(df, SUBSET_MODE)

print(f"\nSubset '{SUBSET_MODE}' rows:", len(df))
print("Unique sessions:", df[GROUP_COL].astype(str).nunique())
print("Unique table types:", df["table_type"].nunique())
if len(df) > 0:
    print(df["surface"].astype(str).value_counts(dropna=False))

if len(df) < 12:
    raise ValueError("Subset too small to train reliably.")

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_square_bbox_from_row(row):
    lefts, rights, tops, bottoms = [], [], [], []

    for roi in ["UL", "UR", "LR", "LL"]:
        x = int(row[f"x_{roi}"])
        y = int(row[f"y_{roi}"])
        r = ROI_SIZES[roi] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts))
    right = int(max(rights))
    top = int(min(tops))
    bottom = int(max(bottoms))
    return left, top, right, bottom

def pil_to_tensor(img):
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.tensor(arr, dtype=torch.float32)

def resize_pil(img, size):
    return img.resize((size, size))

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }

# ---------------------------------------------------------
# SPLIT
# ---------------------------------------------------------
X_dummy = np.zeros((len(df), 1))
y = df[TARGET_COL].values
groups = df[GROUP_COL].astype(str).values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X_dummy, y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

if len(train_df) < 10 or len(test_df) < 3:
    raise ValueError("Train/test split too small for this subset.")

# ---------------------------------------------------------
# SCALE TABULAR FEATURES
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_tab = scaler.fit_transform(train_df[tabular_feature_cols].values)
X_test_tab = scaler.transform(test_df[tabular_feature_cols].values)

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()

for i, col in enumerate(tabular_feature_cols):
    train_df_scaled[col] = X_train_tab[:, i]
    test_df_scaled[col] = X_test_tab[:, i]

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
class HybridSquareROI5Dataset(Dataset):
    def __init__(self, df, tabular_cols, roi_order, roi_sizes,
                 roi_img_size=96, square_img_size=160):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.roi_order = roi_order
        self.roi_sizes = roi_sizes
        self.roi_img_size = roi_img_size
        self.square_img_size = square_img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        roi_tensors = []
        for roi in self.roi_order:
            x = int(row[f"x_{roi}"])
            y = int(row[f"y_{roi}"])
            crop_size = int(self.roi_sizes[roi])

            patch = crop_patch(img, x, y, crop_size)
            patch = resize_pil(patch, self.roi_img_size)
            roi_tensors.append(pil_to_tensor(patch))

        roi_stack = torch.stack(roi_tensors, dim=0)

        left, top, right, bottom = get_square_bbox_from_row(row)
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_patch = resize_pil(square_patch, self.square_img_size)
        square_tensor = pil_to_tensor(square_patch)

        tab = torch.tensor(row[self.tabular_cols].values.astype(np.float32), dtype=torch.float32)
        target = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)

        return {
            "roi_imgs": roi_stack,
            "square_img": square_tensor,
            "tabular": tab,
            "target": target,
            "image_path": row["image_path"],
            "filename": row["filename"],
            "table_type": row["table_type"],
            "surface": row["surface"],
        }

train_ds = HybridSquareROI5Dataset(
    train_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)
test_ds = HybridSquareROI5Dataset(
    test_df_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------
class ROIEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class SquareEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class HybridOptionBModel(nn.Module):
    def __init__(self, n_tabular_features, roi_embed_dim=128, square_embed_dim=128, tab_embed_dim=64):
        super().__init__()
        self.roi_encoder = ROIEncoder(out_dim=roi_embed_dim)
        self.square_encoder = SquareEncoder(out_dim=square_embed_dim)
        self.tab_branch = TabularMLP(n_tabular_features, hidden_dim=128, out_dim=tab_embed_dim)

        fusion_dim = 5 * roi_embed_dim + square_embed_dim + tab_embed_dim
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, roi_imgs, square_img, tabular):
        B, N, C, H, W = roi_imgs.shape
        roi_imgs = roi_imgs.view(B * N, C, H, W)
        roi_emb = self.roi_encoder(roi_imgs)
        roi_emb = roi_emb.view(B, N, -1).flatten(1)

        square_emb = self.square_encoder(square_img)
        tab_emb = self.tab_branch(tabular)

        fused = torch.cat([roi_emb, square_emb, tab_emb], dim=1)
        return self.head(fused).squeeze(1)

model = HybridOptionBModel(n_tabular_features=len(tabular_feature_cols)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# ---------------------------------------------------------
# TRAIN WITH BEST CHECKPOINT BY VAL MAE
# ---------------------------------------------------------
history = []
best_mae = float("inf")
best_epoch = -1
best_state_path = OUT_DIR / "best_model.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(roi_imgs, square_img, tabular)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            roi_imgs = batch["roi_imgs"].to(DEVICE)
            square_img = batch["square_img"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE)

            preds = model(roi_imgs, square_img, tabular).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(batch["target"].numpy().tolist())

    metrics = evaluate_regression(np.array(y_true), np.array(y_pred))
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **metrics
    }
    history.append(row)

    if metrics["mae"] < best_mae:
        best_mae = metrics["mae"]
        best_epoch = epoch
        torch.save(model.state_dict(), best_state_path)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"MAE={row['mae']:.3f} | RMSE={row['rmse']:.3f} | "
        f"R2={row['r2']:.4f} | MAPE={row['mape_percent']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

print(f"\nBest epoch by MAE: {best_epoch} | best MAE={best_mae:.4f}")

# ---------------------------------------------------------
# LOAD BEST MODEL AND EVALUATE
# ---------------------------------------------------------
model.load_state_dict(torch.load(best_state_path, map_location=DEVICE))
model.eval()

final_rows = []

with torch.no_grad():
    for batch in test_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)

        preds = model(roi_imgs, square_img, tabular).cpu().numpy()

        for i in range(len(preds)):
            true_val = float(batch["target"][i].item())
            pred_val = float(preds[i])
            final_rows.append({
                "image_path": batch["image_path"][i],
                "filename": batch["filename"][i],
                "table_type": batch["table_type"][i],
                "surface": batch["surface"][i],
                "target_lux": true_val,
                "pred": pred_val,
                "abs_err": abs(pred_val - true_val),
                "pct_err": abs(pred_val - true_val) / max(abs(true_val), 1e-8) * 100.0,
            })

pred_df = pd.DataFrame(final_rows)
pred_df.to_csv(OUT_DIR / "test_predictions_best_epoch.csv", index=False)

final_metrics = evaluate_regression(pred_df["target_lux"].values, pred_df["pred"].values)
metrics_df = pd.DataFrame([{
    "subset_mode": SUBSET_MODE,
    "model": "Hybrid_OptionB_Square_ROI5_plus_features",
    "best_epoch": best_epoch,
    "n_tabular_features": len(tabular_feature_cols),
    "roi_img_size": ROI_IMG_SIZE,
    "square_img_size": SQUARE_IMG_SIZE,
    **final_metrics
}])
metrics_df.to_csv(OUT_DIR / "final_metrics_best_epoch.csv", index=False)

print("\nFINAL METRICS (BEST EPOCH)")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/hybrid_optionB_subset_runs") / "tables_only"
# change the folder above to match the subset you ran

hist = pd.read_csv(BASE_DIR / "training_history.csv")
pred = pd.read_csv(BASE_DIR / "test_predictions_best_epoch.csv")

plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["mae"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("Option B — validation MAE by epoch")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,6))
true = pred["target_lux"].values
predv = pred["pred"].values
mn = min(true.min(), predv.min())
mx = max(true.max(), predv.max())
plt.scatter(true, predv, alpha=0.55)
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("True lux")
plt.ylabel("Predicted lux")
plt.title("Option B — Predicted vs True")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.hist(pred["abs_err"], bins=25)
plt.xlabel("Absolute error (lux)")
plt.ylabel("Count")
plt.title("Option B — absolute error histogram")
plt.tight_layout()
plt.show()

more features

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageOps, Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
ANNOT_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/annotations/annotation_master_current.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_compare_baseline5x5_vs_richer5x5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
GRID_N = 5
EXTRA_MARGIN = 0

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
CORNER_ROIS = ["UL", "UR", "LR", "LL"]

ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

# =========================================================
# HELPERS
# =========================================================
def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_tight_bbox_from_corner_rois(corner_points_xy, roi_sizes, extra_margin=0):
    lefts, rights, tops, bottoms = [], [], [], []
    for roi_name, (x, y) in corner_points_xy.items():
        r = roi_sizes[roi_name] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    left = int(min(lefts) - extra_margin)
    right = int(max(rights) + extra_margin)
    top = int(min(tops) - extra_margin)
    bottom = int(max(bottoms) + extra_margin)
    return left, top, right, bottom

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="edge")
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def compute_basic_features_from_array(arr):
    arr = arr.astype(np.float32)
    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b
    rgb_sum = r + g + b + 1e-8
    sat_proxy = (np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])) / rgb_sum

    gy, gx = np.gradient(luma)
    grad_mag = np.sqrt(gx**2 + gy**2)

    p10 = np.percentile(luma, 10)
    p50 = np.percentile(luma, 50)
    p90 = np.percentile(luma, 90)

    return {
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
        "mean_luma": float(luma.mean()),
        "std_luma": float(luma.std()),
        "luma_p10": float(p10),
        "luma_p50": float(p50),
        "luma_p90": float(p90),
        "mean_sat_proxy": float(sat_proxy.mean()),
        "std_sat_proxy": float(sat_proxy.std()),
        "grad_mean": float(grad_mag.mean()),
        "grad_std": float(grad_mag.std()),
        "highlight_frac": float(np.mean(luma >= p90)),
        "shadow_frac": float(np.mean(luma <= p10)),
    }

def compute_patch_features(img, x, y, crop_size):
    patch = crop_patch(img, x, y, crop_size)
    return compute_basic_features_from_array(np.asarray(patch))

def compute_grid_features_baseline(patch_img, grid_n=5, prefix="square"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)
            keep = ["mean_luma", "std_luma", "grad_mean", "mean_sat_proxy"]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]
    return feats

def compute_grid_features_richer(patch_img, grid_n=5, prefix="square"):
    arr = np.asarray(patch_img)
    h, w = arr.shape[:2]
    rows = np.linspace(0, h, grid_n + 1, dtype=int)
    cols = np.linspace(0, w, grid_n + 1, dtype=int)

    feats = {}
    cell_means = np.zeros((grid_n, grid_n), dtype=float)

    for i in range(grid_n):
        for j in range(grid_n):
            cell = arr[rows[i]:rows[i+1], cols[j]:cols[j+1], :]
            cell_feats = compute_basic_features_from_array(cell)
            keep = [
                "mean_luma", "std_luma", "grad_mean", "grad_std",
                "mean_sat_proxy", "highlight_frac", "shadow_frac",
                "luma_p10", "luma_p50", "luma_p90",
            ]
            for k in keep:
                feats[f"{prefix}_r{i}c{j}_{k}"] = cell_feats[k]
            cell_means[i, j] = cell_feats["mean_luma"]

    top_mean = float(cell_means[0, :].mean())
    bottom_mean = float(cell_means[-1, :].mean())
    left_mean = float(cell_means[:, 0].mean())
    right_mean = float(cell_means[:, -1].mean())
    center_mean = float(cell_means[grid_n // 2, grid_n // 2])
    corner_mean = float(np.mean([
        cell_means[0, 0], cell_means[0, -1],
        cell_means[-1, 0], cell_means[-1, -1]
    ]))

    feats[f"{prefix}_top_mean_luma"] = top_mean
    feats[f"{prefix}_bottom_mean_luma"] = bottom_mean
    feats[f"{prefix}_left_mean_luma"] = left_mean
    feats[f"{prefix}_right_mean_luma"] = right_mean
    feats[f"{prefix}_center_cell_mean_luma"] = center_mean
    feats[f"{prefix}_corner_cells_mean_luma"] = corner_mean
    feats[f"{prefix}_bottom_minus_top_luma"] = bottom_mean - top_mean
    feats[f"{prefix}_right_minus_left_luma"] = right_mean - left_mean
    feats[f"{prefix}_center_minus_corners_luma"] = center_mean - corner_mean
    feats[f"{prefix}_grid_max_mean_luma"] = float(cell_means.max())
    feats[f"{prefix}_grid_min_mean_luma"] = float(cell_means.min())
    feats[f"{prefix}_grid_range_mean_luma"] = float(cell_means.max() - cell_means.min())
    feats[f"{prefix}_grid_std_mean_luma"] = float(cell_means.std())

    return feats

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def make_subsets(df):
    return {
        "all_5roi_data": df.copy(),
        "white_paper_only": df[df["surface"] == "white_paper"].copy(),
        "tables_only": df[df["surface"] == "table"].copy(),
        "white_plus_tables": df[df["surface"].isin(["white_paper", "table"])].copy(),
    }

# =========================================================
# LOAD BASE IMAGE-LEVEL ROWS
# =========================================================
ann = pd.read_csv(ANNOT_CSV)
ann = ann[ann["label_type"] == "5roi"].copy()

base_rows = []
for image_path, g in ann.groupby("image_path"):
    if set(g["roi_name"]) != set(ROI_ORDER):
        continue

    g = g.set_index("roi_name").loc[ROI_ORDER].reset_index()

    rec = {
        "image_path": image_path,
        "filename": g.iloc[0]["filename"],
        "session": str(g.iloc[0]["session"]),
        "table_type": str(g.iloc[0]["table_type"]),
        "surface": str(g.iloc[0]["surface"]).lower(),
        "target_lux": float(g[g["roi_name"] == "C"]["lux"].iloc[0]),
    }

    for _, row in g.iterrows():
        roi = row["roi_name"]
        rec[f"x_{roi}"] = int(row["x"])
        rec[f"y_{roi}"] = int(row["y"])

    base_rows.append(rec)

base_df = pd.DataFrame(base_rows)
print("Base image-level rows:", len(base_df))

# =========================================================
# BUILD BOTH FEATURE SETS
# =========================================================
feature_sets = {
    "baseline5x5": compute_grid_features_baseline,
    "richer5x5": compute_grid_features_richer,
}

all_metrics = []

for feature_set_name, grid_func in feature_sets.items():
    print(f"\n================ FEATURE SET = {feature_set_name} ================\n")
    rows = []

    for _, row in base_df.iterrows():
        try:
            img = Image.open(row["image_path"])
            img = ImageOps.exif_transpose(img).convert("RGB")

            rec = row.to_dict()
            rec["feature_set"] = feature_set_name

            roi_luma_means = {}
            roi_grad_means = {}

            # per-ROI features
            for roi in ROI_ORDER:
                x = row[f"x_{roi}"]
                y = row[f"y_{roi}"]
                feats = compute_patch_features(img, x, y, ROI_SIZES[roi])
                for k, v in feats.items():
                    rec[f"{roi}_{k}"] = v
                roi_luma_means[roi] = feats["mean_luma"]
                roi_grad_means[roi] = feats["grad_mean"]

            # ROI relations
            corner_lumas = np.array([roi_luma_means[r] for r in CORNER_ROIS], dtype=float)
            corner_grads = np.array([roi_grad_means[r] for r in CORNER_ROIS], dtype=float)

            rec["corner_mean_luma"] = float(corner_lumas.mean())
            rec["corner_std_luma"] = float(corner_lumas.std())
            rec["corner_min_luma"] = float(corner_lumas.min())
            rec["corner_max_luma"] = float(corner_lumas.max())
            rec["corner_range_luma"] = float(corner_lumas.max() - corner_lumas.min())

            rec["C_minus_corner_mean_luma"] = float(roi_luma_means["C"] - corner_lumas.mean())
            rec["C_over_corner_mean_luma"] = float(roi_luma_means["C"] / (corner_lumas.mean() + 1e-8))

            rec["corner_mean_grad"] = float(corner_grads.mean())
            rec["corner_std_grad"] = float(corner_grads.std())
            rec["C_minus_corner_mean_grad"] = float(roi_grad_means["C"] - corner_grads.mean())

            for roi in CORNER_ROIS:
                rec[f"C_minus_{roi}_luma"] = float(roi_luma_means["C"] - roi_luma_means[roi])

            # square patch
            corner_points_xy = {
                "UL": (row["x_UL"], row["y_UL"]),
                "UR": (row["x_UR"], row["y_UR"]),
                "LR": (row["x_LR"], row["y_LR"]),
                "LL": (row["x_LL"], row["y_LL"]),
            }

            left, top, right, bottom = get_tight_bbox_from_corner_rois(
                corner_points_xy, ROI_SIZES, extra_margin=EXTRA_MARGIN
            )
            square_patch = crop_bbox(img, left, top, right, bottom)

            square_basic = compute_basic_features_from_array(np.asarray(square_patch))
            for k, v in square_basic.items():
                rec[f"square_{k}"] = v

            square_grid = grid_func(square_patch, grid_n=GRID_N, prefix="square")
            rec.update(square_grid)

            rec["C_minus_square_mean_luma"] = float(rec["C_mean_luma"] - rec["square_mean_luma"])
            rec["corner_mean_minus_square_mean_luma"] = float(rec["corner_mean_luma"] - rec["square_mean_luma"])
            rec["C_over_square_mean_luma"] = float(rec["C_mean_luma"] / (rec["square_mean_luma"] + 1e-8))

            rows.append(rec)

        except Exception as e:
            print("Skipped:", row["image_path"], e)

    df = pd.DataFrame(rows)
    print("Rows built:", len(df))
    print("Columns:", len(df.columns))

    df.to_csv(OUT_DIR / f"extracted_features__{feature_set_name}.csv", index=False)

    exclude_cols = {
        "image_path", "filename", "session", "table_type", "surface",
        "target_lux", "feature_set"
    }
    exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
    exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print("Number of feature columns:", len(feature_cols))

    subsets = make_subsets(df)

    for subset_name, df_sub in subsets.items():
        print(f"=== {subset_name} ===")
        print("Rows:", len(df_sub))
        if len(df_sub) < 20:
            continue

        X = df_sub[feature_cols].values
        y = df_sub["target_lux"].values
        groups = df_sub["session"].astype(str).values

        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))

        train_df = df_sub.iloc[train_idx].copy()
        test_df = df_sub.iloc[test_idx].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df["target_lux"].values
        X_test = test_df[feature_cols].values
        y_test = test_df["target_lux"].values

        model = ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        rowm = {
            "subset": subset_name,
            "feature_set": feature_set_name,
            "model": "ExtraTrees_compare_5x5",
            "grid_n": GRID_N,
            "n_features": len(feature_cols),
            "n_train": len(train_df),
            "n_test": len(test_df),
            "mae": mean_absolute_error(y_test, y_pred),
            "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "r2": r2_score(y_test, y_pred),
            "mape_percent": mape_percent(y_test, y_pred),
        }
        all_metrics.append(rowm)
        print(pd.DataFrame([rowm]).to_string(index=False))

        pred_df = test_df.copy()
        pred_df["pred"] = y_pred
        pred_df["abs_err"] = np.abs(pred_df["target_lux"].values - y_pred)
        pred_df["pct_err"] = np.abs(
            (pred_df["target_lux"].values - y_pred) / np.maximum(np.abs(pred_df["target_lux"].values), 1e-8)
        ) * 100.0
        pred_df.to_csv(OUT_DIR / f"{feature_set_name}__{subset_name}__predictions.csv", index=False)

        fi_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        fi_df.to_csv(OUT_DIR / f"{feature_set_name}__{subset_name}__feature_importance.csv", index=False)

metrics_df = pd.DataFrame(all_metrics).sort_values(["subset", "mae"]).reset_index(drop=True)
metrics_df.to_csv(OUT_DIR / "metrics_compare.csv", index=False)

print("\n========== FINAL METRICS ==========")
print(metrics_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "metrics_compare.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_compare_baseline5x5_vs_richer5x5")
df = pd.read_csv(BASE_DIR / "metrics_compare.csv")

print(df.to_string(index=False))

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, subset in zip(axes, subset_order):
    sub = df[df["subset"] == subset].copy()
    sub["feature_set"] = pd.Categorical(sub["feature_set"], categories=["baseline5x5", "richer5x5"], ordered=True)
    sub = sub.sort_values("feature_set")

    ax.bar(sub["feature_set"], sub["mae"])
    ax.set_title(subset)
    ax.set_ylabel("MAE (lux)")

plt.tight_layout()
plt.savefig(BASE_DIR / "compare__mae.png", dpi=220, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, subset in zip(axes, subset_order):
    sub = df[df["subset"] == subset].copy()
    sub["feature_set"] = pd.Categorical(sub["feature_set"], categories=["baseline5x5", "richer5x5"], ordered=True)
    sub = sub.sort_values("feature_set")

    ax.bar(sub["feature_set"], sub["mape_percent"])
    ax.set_title(subset)
    ax.set_ylabel("MAPE (%)")

plt.tight_layout()
plt.savefig(BASE_DIR / "compare__mape.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# FAIR COMPARISON:
# Option B (Square + 5 ROI + tabular features)
# vs
# ExtraTrees baseline 5x5
#
# Subset: white_plus_tables
#
# This script:
# 1) builds ONE shared train/test split by session
# 2) trains ExtraTrees baseline5x5 on that exact split
# 3) trains Option B on that exact split
# 4) saves metrics, predictions, and comparison plots
# =========================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
RICH_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_richer_features_plus_square/extracted_features_master.csv")
BASELINE5X5_CSV = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_compare_baseline5x5_vs_richer5x5/extracted_features__baseline5x5.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/compare_optionB_vs_extratrees_white_plus_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_MODE = "white_plus_tables"

SEED = 42
TEST_SIZE = 0.2

# ExtraTrees
N_ESTIMATORS = 500

# Option B
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 12
NUM_EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 1e-4

ROI_IMG_SIZE = 96
SQUARE_IMG_SIZE = 160

ROI_ORDER = ["C", "UL", "UR", "LR", "LL"]
ROI_SIZES = {
    "C": 256,
    "UL": 128,
    "UR": 128,
    "LR": 128,
    "LL": 128,
}

TARGET_COL = "target_lux"
GROUP_COL = "session"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def apply_subset(df_in, subset_mode):
    df_sub = df_in.copy()
    surface_lower = df_sub["surface"].astype(str).str.strip().str.lower()

    if subset_mode == "all_5roi_data":
        pass
    elif subset_mode == "white_paper_only":
        df_sub = df_sub[surface_lower == "white_paper"].copy()
    elif subset_mode == "tables_only":
        df_sub = df_sub[surface_lower == "table"].copy()
    elif subset_mode == "white_plus_tables":
        df_sub = df_sub[surface_lower.isin(["white_paper", "table"])].copy()
    else:
        raise ValueError(f"Unsupported subset: {subset_mode}")

    return df_sub.reset_index(drop=True)

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "mape_percent": mape,
    }

def crop_patch(img, x, y, crop_size):
    half = crop_size // 2
    left = int(round(x - half))
    top = int(round(y - half))
    right = left + crop_size
    bottom = top + crop_size

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def crop_bbox(img, left, top, right, bottom):
    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - img.width)
    pad_bottom = max(0, bottom - img.height)

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.array(img)
        arr = np.pad(
            arr,
            ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
            mode="edge"
        )
        img = Image.fromarray(arr)
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top

    return img.crop((left, top, right, bottom))

def get_square_bbox_from_row(row):
    lefts, rights, tops, bottoms = [], [], [], []
    for roi in ["UL", "UR", "LR", "LL"]:
        x = int(row[f"x_{roi}"])
        y = int(row[f"y_{roi}"])
        r = ROI_SIZES[roi] // 2
        lefts.append(x - r)
        rights.append(x + r)
        tops.append(y - r)
        bottoms.append(y + r)

    return int(min(lefts)), int(min(tops)), int(max(rights)), int(max(bottoms))

def pil_to_tensor(img):
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.tensor(arr, dtype=torch.float32)

def resize_pil(img, size):
    return img.resize((size, size))

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
rich_df = pd.read_csv(RICH_CSV)
baseline_df = pd.read_csv(BASELINE5X5_CSV)

rich_df = apply_subset(rich_df, SUBSET_MODE)
baseline_df = apply_subset(baseline_df, SUBSET_MODE)

print("Rich rows:", len(rich_df))
print("Baseline rows:", len(baseline_df))

# Use image_path as the key to align splits across both tables
common_paths = set(rich_df["image_path"]).intersection(set(baseline_df["image_path"]))
rich_df = rich_df[rich_df["image_path"].isin(common_paths)].copy().reset_index(drop=True)
baseline_df = baseline_df[baseline_df["image_path"].isin(common_paths)].copy().reset_index(drop=True)

print("Common rows after alignment:", len(rich_df), len(baseline_df))

# Make sure no duplicates by image_path
rich_df = rich_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)
baseline_df = baseline_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

# Shared split from rich_df
X_dummy = np.zeros((len(rich_df), 1))
y = rich_df[TARGET_COL].values
groups = rich_df[GROUP_COL].astype(str).values

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X_dummy, y, groups=groups))

rich_train = rich_df.iloc[train_idx].reset_index(drop=True)
rich_test = rich_df.iloc[test_idx].reset_index(drop=True)

train_paths = set(rich_train["image_path"])
test_paths = set(rich_test["image_path"])

baseline_train = baseline_df[baseline_df["image_path"].isin(train_paths)].copy().reset_index(drop=True)
baseline_test = baseline_df[baseline_df["image_path"].isin(test_paths)].copy().reset_index(drop=True)

print("Shared split sizes")
print("Rich train/test:", len(rich_train), len(rich_test))
print("Baseline train/test:", len(baseline_train), len(baseline_test))

# ---------------------------------------------------------
# EXTRATREES BASELINE 5x5
# ---------------------------------------------------------
exclude_cols = {
    "image_path", "filename", "session", "table_type", "surface",
    "target_lux", "feature_set"
}
exclude_cols.update({f"x_{r}" for r in ROI_ORDER})
exclude_cols.update({f"y_{r}" for r in ROI_ORDER})

baseline_feature_cols = [c for c in baseline_train.columns if c not in exclude_cols]
print("ExtraTrees feature count:", len(baseline_feature_cols))

X_train_et = baseline_train[baseline_feature_cols].values
y_train_et = baseline_train[TARGET_COL].values
X_test_et = baseline_test[baseline_feature_cols].values
y_test_et = baseline_test[TARGET_COL].values

et_model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    random_state=SEED,
    n_jobs=-1
)
et_model.fit(X_train_et, y_train_et)
et_pred = et_model.predict(X_test_et)

et_metrics = evaluate_regression(y_test_et, et_pred)
et_metrics_row = {
    "subset_mode": SUBSET_MODE,
    "model": "ExtraTrees_baseline5x5",
    "n_features": len(baseline_feature_cols),
    **et_metrics
}
print("\nEXTRATREES METRICS")
print(pd.DataFrame([et_metrics_row]).to_string(index=False))

et_pred_df = baseline_test.copy()
et_pred_df["pred"] = et_pred
et_pred_df["abs_err"] = np.abs(et_pred_df[TARGET_COL].values - et_pred)
et_pred_df["pct_err"] = np.abs(
    (et_pred_df[TARGET_COL].values - et_pred) / np.maximum(np.abs(et_pred_df[TARGET_COL].values), 1e-8)
) * 100.0
et_pred_df.to_csv(OUT_DIR / "extratrees_test_predictions.csv", index=False)

et_fi_df = pd.DataFrame({
    "feature": baseline_feature_cols,
    "importance": et_model.feature_importances_
}).sort_values("importance", ascending=False)
et_fi_df.to_csv(OUT_DIR / "extratrees_feature_importance.csv", index=False)

# ---------------------------------------------------------
# OPTION B DATA PREP
# ---------------------------------------------------------
tabular_feature_cols = [
    "C_mean_luma", "C_std_luma", "C_grad_mean", "C_grad_std",
    "UL_mean_luma", "UR_mean_luma", "LR_mean_luma", "LL_mean_luma",
    "UL_grad_mean", "UR_grad_mean", "LR_grad_mean", "LL_grad_mean",
    "corner_mean_luma", "corner_std_luma", "corner_range_luma",
    "C_minus_corner_mean_luma", "C_over_corner_mean_luma",
    "C_minus_UL_luma", "C_minus_UR_luma", "C_minus_LR_luma", "C_minus_LL_luma",
    "square_mean_luma", "square_std_luma", "square_grad_mean", "square_grad_std",
    "square_mean_sat_proxy",
    "square_top_mean_luma", "square_bottom_mean_luma",
    "square_left_mean_luma", "square_right_mean_luma",
    "square_center_cell_mean_luma", "square_corner_cells_mean_luma",
    "square_bottom_minus_top_luma", "square_right_minus_left_luma",
    "square_center_minus_corners_luma",
    "square_grid_range_mean_luma", "square_grid_std_mean_luma",
]
tabular_feature_cols = [c for c in tabular_feature_cols if c in rich_train.columns]
print("Option B tabular feature count:", len(tabular_feature_cols))

scaler = StandardScaler()
X_train_tab = scaler.fit_transform(rich_train[tabular_feature_cols].values)
X_test_tab = scaler.transform(rich_test[tabular_feature_cols].values)

rich_train_scaled = rich_train.copy()
rich_test_scaled = rich_test.copy()
for i, col in enumerate(tabular_feature_cols):
    rich_train_scaled[col] = X_train_tab[:, i]
    rich_test_scaled[col] = X_test_tab[:, i]

# ---------------------------------------------------------
# OPTION B DATASET
# ---------------------------------------------------------
class HybridSquareROI5Dataset(Dataset):
    def __init__(self, df, tabular_cols, roi_order, roi_sizes,
                 roi_img_size=96, square_img_size=160):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_cols
        self.roi_order = roi_order
        self.roi_sizes = roi_sizes
        self.roi_img_size = roi_img_size
        self.square_img_size = square_img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["image_path"])
        img = ImageOps.exif_transpose(img).convert("RGB")

        roi_tensors = []
        for roi in self.roi_order:
            x = int(row[f"x_{roi}"])
            y = int(row[f"y_{roi}"])
            crop_size = int(self.roi_sizes[roi])

            patch = crop_patch(img, x, y, crop_size)
            patch = resize_pil(patch, self.roi_img_size)
            roi_tensors.append(pil_to_tensor(patch))

        roi_stack = torch.stack(roi_tensors, dim=0)

        left, top, right, bottom = get_square_bbox_from_row(row)
        square_patch = crop_bbox(img, left, top, right, bottom)
        square_patch = resize_pil(square_patch, self.square_img_size)
        square_tensor = pil_to_tensor(square_patch)

        tab = torch.tensor(row[self.tabular_cols].values.astype(np.float32), dtype=torch.float32)
        target = torch.tensor(float(row[TARGET_COL]), dtype=torch.float32)

        return {
            "roi_imgs": roi_stack,
            "square_img": square_tensor,
            "tabular": tab,
            "target": target,
            "image_path": row["image_path"],
            "filename": row["filename"],
        }

train_ds = HybridSquareROI5Dataset(
    rich_train_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)
test_ds = HybridSquareROI5Dataset(
    rich_test_scaled, tabular_feature_cols, ROI_ORDER, ROI_SIZES,
    roi_img_size=ROI_IMG_SIZE, square_img_size=SQUARE_IMG_SIZE
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# OPTION B MODEL
# ---------------------------------------------------------
class ROIEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class SquareEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.fc(x)

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class HybridOptionBModel(nn.Module):
    def __init__(self, n_tabular_features, roi_embed_dim=128, square_embed_dim=128, tab_embed_dim=64):
        super().__init__()
        self.roi_encoder = ROIEncoder(out_dim=roi_embed_dim)
        self.square_encoder = SquareEncoder(out_dim=square_embed_dim)
        self.tab_branch = TabularMLP(n_tabular_features, hidden_dim=128, out_dim=tab_embed_dim)

        fusion_dim = 5 * roi_embed_dim + square_embed_dim + tab_embed_dim
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, roi_imgs, square_img, tabular):
        B, N, C, H, W = roi_imgs.shape
        roi_imgs = roi_imgs.view(B * N, C, H, W)
        roi_emb = self.roi_encoder(roi_imgs)
        roi_emb = roi_emb.view(B, N, -1).flatten(1)

        square_emb = self.square_encoder(square_img)
        tab_emb = self.tab_branch(tabular)

        fused = torch.cat([roi_emb, square_emb, tab_emb], dim=1)
        return self.head(fused).squeeze(1)

optionb_model = HybridOptionBModel(n_tabular_features=len(tabular_feature_cols)).to(DEVICE)
optimizer = torch.optim.Adam(optionb_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

# ---------------------------------------------------------
# TRAIN OPTION B
# ---------------------------------------------------------
history = []
best_mae = float("inf")
best_epoch = -1
best_state_path = OUT_DIR / "optionb_best_model.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    optionb_model.train()
    train_losses = []

    for batch in train_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        pred = optionb_model(roi_imgs, square_img, tabular)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    optionb_model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_loader:
            roi_imgs = batch["roi_imgs"].to(DEVICE)
            square_img = batch["square_img"].to(DEVICE)
            tabular = batch["tabular"].to(DEVICE)

            preds = optionb_model(roi_imgs, square_img, tabular).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(batch["target"].numpy().tolist())

    metrics = evaluate_regression(np.array(y_true), np.array(y_pred))
    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **metrics
    }
    history.append(row)

    if metrics["mae"] < best_mae:
        best_mae = metrics["mae"]
        best_epoch = epoch
        torch.save(optionb_model.state_dict(), best_state_path)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"MAE={row['mae']:.3f} | RMSE={row['rmse']:.3f} | "
        f"R2={row['r2']:.4f} | MAPE={row['mape_percent']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "optionb_training_history.csv", index=False)

optionb_model.load_state_dict(torch.load(best_state_path, map_location=DEVICE))
optionb_model.eval()

final_rows = []
with torch.no_grad():
    for batch in test_loader:
        roi_imgs = batch["roi_imgs"].to(DEVICE)
        square_img = batch["square_img"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)

        preds = optionb_model(roi_imgs, square_img, tabular).cpu().numpy()

        for i in range(len(preds)):
            true_val = float(batch["target"][i].item())
            pred_val = float(preds[i])
            final_rows.append({
                "image_path": batch["image_path"][i],
                "filename": batch["filename"][i],
                "target_lux": true_val,
                "pred": pred_val,
                "abs_err": abs(pred_val - true_val),
                "pct_err": abs(pred_val - true_val) / max(abs(true_val), 1e-8) * 100.0,
            })

optionb_pred_df = pd.DataFrame(final_rows)
optionb_pred_df.to_csv(OUT_DIR / "optionb_test_predictions_best_epoch.csv", index=False)

optionb_metrics = evaluate_regression(optionb_pred_df["target_lux"].values, optionb_pred_df["pred"].values)
optionb_metrics_row = {
    "subset_mode": SUBSET_MODE,
    "model": "Hybrid_OptionB_Square_ROI5_plus_features",
    "best_epoch": best_epoch,
    "n_tabular_features": len(tabular_feature_cols),
    "roi_img_size": ROI_IMG_SIZE,
    "square_img_size": SQUARE_IMG_SIZE,
    **optionb_metrics
}
print("\nOPTION B METRICS")
print(pd.DataFrame([optionb_metrics_row]).to_string(index=False))

# ---------------------------------------------------------
# SAVE COMPARISON TABLE
# ---------------------------------------------------------
compare_df = pd.DataFrame([
    et_metrics_row,
    optionb_metrics_row,
])
compare_df.to_csv(OUT_DIR / "compare_metrics.csv", index=False)

print("\nCOMPARISON")
print(compare_df.to_string(index=False))
print("\nSaved to:", OUT_DIR)

# ---------------------------------------------------------
# PLOTS
# ---------------------------------------------------------
import matplotlib.pyplot as plt

# bar chart
plot_df = compare_df.copy()
plot_df["label"] = plot_df["model"]

for metric, ylabel in [
    ("mae", "MAE (lux)"),
    ("rmse", "RMSE (lux)"),
    ("r2", "R²"),
    ("mape_percent", "MAPE (%)"),
]:
    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["label"], plot_df[metric])
    plt.ylabel(ylabel)
    plt.title(f"{SUBSET_MODE}: ExtraTrees vs Option B")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"compare__{metric}.png", dpi=220, bbox_inches="tight")
    plt.show()

# predicted vs true side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ET
true = et_pred_df[TARGET_COL].values
pred = et_pred_df["pred"].values
mn = min(true.min(), pred.min())
mx = max(true.max(), pred.max())
axes[0].scatter(true, pred, alpha=0.55)
axes[0].plot([mn, mx], [mn, mx], linestyle="--")
axes[0].set_title("ExtraTrees baseline5x5")
axes[0].set_xlabel("True lux")
axes[0].set_ylabel("Predicted lux")

# Option B
true = optionb_pred_df["target_lux"].values
pred = optionb_pred_df["pred"].values
mn = min(true.min(), pred.min())
mx = max(true.max(), pred.max())
axes[1].scatter(true, pred, alpha=0.55)
axes[1].plot([mn, mx], [mn, mx], linestyle="--")
axes[1].set_title("Option B")
axes[1].set_xlabel("True lux")
axes[1].set_ylabel("Predicted lux")

plt.tight_layout()
plt.savefig(OUT_DIR / "compare__pred_vs_true.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================================================
# CONFIG
# =========================================================
CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/extratrees_compare_baseline5x5_vs_richer5x5/extracted_features__baseline5x5.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/baseline_model_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.2
RANDOM_STATE = 42

SUBSETS_TO_RUN = [
    "white_paper_only",
    "tables_only",
    "white_plus_tables",
    "all_5roi_data",
]

TARGET_COL = "target_lux"
GROUP_COL = "session"

# =========================================================
# HELPERS
# =========================================================
def apply_subset(df_in, subset_mode):
    df_sub = df_in.copy()
    surface_lower = df_sub["surface"].astype(str).str.strip().str.lower()

    if subset_mode == "all_5roi_data":
        pass
    elif subset_mode == "white_paper_only":
        df_sub = df_sub[surface_lower == "white_paper"].copy()
    elif subset_mode == "tables_only":
        df_sub = df_sub[surface_lower == "table"].copy()
    elif subset_mode == "white_plus_tables":
        df_sub = df_sub[surface_lower.isin(["white_paper", "table"])].copy()
    else:
        raise ValueError(f"Unknown subset: {subset_mode}")

    return df_sub.reset_index(drop=True)

def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def evaluate(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
        "mape_percent": mape_percent(y_true, y_pred),
    }

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))

exclude_cols = {
    "image_path", "filename", "session", "table_type", "surface",
    "target_lux", "feature_set"
}
exclude_cols.update({c for c in df.columns if c.startswith("x_") or c.startswith("y_")})

feature_cols = [c for c in df.columns if c not in exclude_cols]
print("Feature count:", len(feature_cols))

# =========================================================
# MODELS
# =========================================================
models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "PolynomialDeg2": Pipeline([
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
}

all_results = []

# =========================================================
# RUN SUBSETS
# =========================================================
for subset_name in SUBSETS_TO_RUN:
    df_sub = apply_subset(df, subset_name)

    print(f"\n=== {subset_name} ===")
    print("Rows:", len(df_sub))
    print("Unique sessions:", df_sub[GROUP_COL].astype(str).nunique())

    if len(df_sub) < 20:
        print("Skipping: too few rows")
        continue

    X = df_sub[feature_cols].values
    y = df_sub[TARGET_COL].values
    groups = df_sub[GROUP_COL].astype(str).values

    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train = df_sub.iloc[train_idx][feature_cols].values
    y_train = df_sub.iloc[train_idx][TARGET_COL].values
    X_test = df_sub.iloc[test_idx][feature_cols].values
    y_test = df_sub.iloc[test_idx][TARGET_COL].values

    print("Train rows:", len(train_idx))
    print("Test rows:", len(test_idx))

    for model_name, model in models.items():
        print("Running:", model_name)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = evaluate(y_test, y_pred)
        row = {
            "subset": subset_name,
            "model": model_name,
            "n_features": len(feature_cols),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            **metrics
        }
        all_results.append(row)

        pred_df = df_sub.iloc[test_idx][["image_path", "filename", "session", "table_type", "surface", TARGET_COL]].copy()
        pred_df["pred"] = y_pred
        pred_df["abs_err"] = np.abs(pred_df[TARGET_COL].values - y_pred)
        pred_df["pct_err"] = np.abs(
            (pred_df[TARGET_COL].values - y_pred) / np.maximum(np.abs(pred_df[TARGET_COL].values), 1e-8)
        ) * 100.0
        pred_df.to_csv(OUT_DIR / f"{subset_name}__{model_name}__predictions.csv", index=False)

results_df = pd.DataFrame(all_results).sort_values(["subset", "mae"]).reset_index(drop=True)
results_df.to_csv(OUT_DIR / "baseline_model_comparison_metrics.csv", index=False)

print("\n========== FINAL RESULTS ==========")
print(results_df.to_string(index=False))
print("\nSaved to:", OUT_DIR / "baseline_model_comparison_metrics.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/baseline_model_comparison/baseline_model_comparison_metrics.csv")
OUT_DIR = Path("/content/drive/MyDrive/AI_Lux_Project/Experiments_3/baseline_model_comparison")

df = pd.read_csv(CSV_PATH)

subset_order = ["white_paper_only", "tables_only", "white_plus_tables", "all_5roi_data"]

for metric, ylabel in [
    ("mae", "MAE (lux)"),
    ("mape_percent", "MAPE (%)"),
    ("r2", "R²"),
]:
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes = axes.flatten()

    for ax, subset in zip(axes, subset_order):
        sub = df[df["subset"] == subset].copy()
        sub = sub.sort_values("mae")

        ax.bar(sub["model"], sub[metric])
        ax.set_title(subset)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"baseline_compare__{metric}.png", dpi=220, bbox_inches="tight")
    plt.show()